In [92]:
# Create the master HAM10000 benchmark table for all five official models

import pandas as pd

ham10000_benchmark = pd.DataFrame([
    {
        "Model": "AlexNet",
        "Accuracy": 0.845539,
        "Precision": 0.385965,
        "Recall": 0.658683,
        "F1": 0.486726,
        "ROC-AUC": 0.870407
    },
    {
        "Model": "VGG16",
        "Accuracy": 0.856192,
        "Precision": 0.415225,
        "Recall": 0.718563,
        "F1": 0.526316,
        "ROC-AUC": 0.896064
    },
    {
        "Model": "VGG19",
        "Accuracy": 0.903462,
        "Precision": 0.568750,
        "Recall": 0.544910,
        "F1": 0.556575,
        "ROC-AUC": 0.872484
    },
    {
        "Model": "ResNet50",
        "Accuracy": 0.881491,
        "Precision": 0.477178,
        "Recall": 0.688623,
        "F1": 0.563725,
        "ROC-AUC": 0.908995
    },
    {
        "Model": "EfficientNet-B0",
        "Accuracy": 0.895473,
        "Precision": 0.534722,
        "Recall": 0.461078,
        "F1": 0.495177,
        "ROC-AUC": 0.888434
    }
])

print("=" * 80)
print("HAM10000 — FIVE-MODEL BENCHMARK")
print("=" * 80)

display(
    ham10000_benchmark.style.format({
        "Accuracy": "{:.4f}",
        "Precision": "{:.4f}",
        "Recall": "{:.4f}",
        "F1": "{:.4f}",
        "ROC-AUC": "{:.4f}"
    })
)

print("\nBest model by F1:")
print(
    ham10000_benchmark.loc[
        ham10000_benchmark["F1"].idxmax(), "Model"
    ]
)

print("\nBest model by ROC-AUC:")
print(
    ham10000_benchmark.loc[
        ham10000_benchmark["ROC-AUC"].idxmax(), "Model"
    ]
)

HAM10000 — FIVE-MODEL BENCHMARK


,Model,Accuracy,Precision,Recall,F1,ROC-AUC
0,AlexNet,0.8455,0.3860,0.6587,0.4867,0.8704
1,VGG16,0.8562,0.4152,0.7186,0.5263,0.8961
2,VGG19,0.9035,0.5687,0.5449,0.5566,0.8725
3,ResNet50,0.8815,0.4772,0.6886,0.5637,0.9090
4,EfficientNet-B0,0.8955,0.5347,0.4611,0.4952,0.8884



Best model by F1:
ResNet50

Best model by ROC-AUC:
ResNet50


In [70]:
# Verify the ResNet50 training environment and required data loaders

print("Device:", DEVICE)
print("Number of GPUs:", torch.cuda.device_count())
print("Train batches:", len(balanced_train_loader))
print("Validation batches:", len(val_loader))
print("Test batches:", len(test_loader))

print("\nResNet50 setup verification complete.")

Device: cuda
Number of GPUs: 2
Train batches: 219
Validation batches: 48
Test batches: 47

ResNet50 setup verification complete.


In [71]:
# Initialize the pretrained ResNet50 model for binary melanoma classification

import torch
import torch.nn as nn
from torchvision import models

resnet50_model = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)

resnet50_model.fc = nn.Linear(
    resnet50_model.fc.in_features,
    NUM_CLASSES
)

resnet50_model = resnet50_model.to(DEVICE)

if torch.cuda.device_count() > 1:
    resnet50_model = nn.DataParallel(resnet50_model)

resnet50_param_count = sum(
    p.numel() for p in resnet50_model.parameters()
)

print("ResNet50 initialized successfully.")
print(f"Parameters: {resnet50_param_count:,}")
print(f"Device: {DEVICE}")
print(f"GPU count: {torch.cuda.device_count()}")

Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 195MB/s] 


ResNet50 initialized successfully.
Parameters: 23,512,130
Device: cuda
GPU count: 2


In [72]:
# Configure the ResNet50 loss function, optimizer, and training settings

resnet50_criterion = nn.CrossEntropyLoss()

resnet50_optimizer = torch.optim.AdamW(
    resnet50_model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY
)

resnet50_scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    resnet50_optimizer,
    mode="max",
    factor=0.5,
    patience=2
)

print("ResNet50 training configuration ready.")
print(f"Learning rate: {LEARNING_RATE}")
print(f"Weight decay: {WEIGHT_DECAY}")
print("Loss: CrossEntropyLoss")
print("Optimizer: AdamW")
print("Scheduler: ReduceLROnPlateau")

ResNet50 training configuration ready.
Learning rate: 0.0001
Weight decay: 1e-05
Loss: CrossEntropyLoss
Optimizer: AdamW
Scheduler: ReduceLROnPlateau


In [73]:
# Train ResNet50 using balanced sampling and validation-F1 model selection

import time
import copy

resnet50_checkpoint_path = "/kaggle/working/ham10000_resnet50_balanced_best.pth"

resnet50_history = {
    "train_loss": [],
    "val_loss": [],
    "val_accuracy": [],
    "val_precision": [],
    "val_recall": [],
    "val_f1": [],
    "val_roc_auc": []
}

resnet50_best_f1 = -1.0
resnet50_best_epoch = 0
resnet50_epochs_without_improvement = 0

training_start = time.time()

for epoch in range(1, MAX_EPOCHS + 1):
    epoch_start = time.time()

    resnet50_model.train()
    running_loss = 0.0
    sample_count = 0

    for images, labels in balanced_train_loader:
        images = images.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)

        resnet50_optimizer.zero_grad(set_to_none=True)

        outputs = resnet50_model(images)
        loss = resnet50_criterion(outputs, labels)

        loss.backward()
        resnet50_optimizer.step()

        batch_size = labels.size(0)
        running_loss += loss.item() * batch_size
        sample_count += batch_size

    train_loss = running_loss / sample_count

    resnet50_val_results = evaluate_model(
        resnet50_model,
        val_loader,
        resnet50_criterion,
        DEVICE
    )

    val_loss = resnet50_val_results["loss"]
    val_accuracy = resnet50_val_results["accuracy"]
    val_precision = resnet50_val_results["precision"]
    val_recall = resnet50_val_results["recall"]
    val_f1 = resnet50_val_results["f1"]
    val_roc_auc = resnet50_val_results["roc_auc"]

    resnet50_history["train_loss"].append(train_loss)
    resnet50_history["val_loss"].append(val_loss)
    resnet50_history["val_accuracy"].append(val_accuracy)
    resnet50_history["val_precision"].append(val_precision)
    resnet50_history["val_recall"].append(val_recall)
    resnet50_history["val_f1"].append(val_f1)
    resnet50_history["val_roc_auc"].append(val_roc_auc)

    resnet50_scheduler.step(val_f1)

    elapsed = time.time() - epoch_start

    print(
        f"Epoch {epoch:02d}/{MAX_EPOCHS} | "
        f"Train Loss {train_loss:.4f} | "
        f"Val Loss {val_loss:.4f} | "
        f"Val Acc {val_accuracy:.4f} | "
        f"Val F1 {val_f1:.4f} | "
        f"Val AUC {val_roc_auc:.4f} | "
        f"Time {elapsed:.1f}s"
    )

    if val_f1 > resnet50_best_f1:
        resnet50_best_f1 = val_f1
        resnet50_best_epoch = epoch
        resnet50_epochs_without_improvement = 0

        torch.save(
            {
                "epoch": epoch,
                "model_state_dict": resnet50_model.state_dict(),
                "optimizer_state_dict": resnet50_optimizer.state_dict(),
                "scheduler_state_dict": resnet50_scheduler.state_dict(),
                "best_val_f1": resnet50_best_f1,
                "history": copy.deepcopy(resnet50_history)
            },
            resnet50_checkpoint_path
        )

        print("  Best checkpoint saved.")

    else:
        resnet50_epochs_without_improvement += 1

    if resnet50_epochs_without_improvement >= EARLY_STOPPING_PATIENCE:
        print(
            f"Early stopping triggered after "
            f"{EARLY_STOPPING_PATIENCE} epochs without improvement."
        )
        break

total_training_time = time.time() - training_start

print("\nResNet50 training completed.")
print(f"Best epoch: {resnet50_best_epoch}")
print(f"Best validation F1: {resnet50_best_f1:.4f}")
print(f"Total training time: {total_training_time / 60:.2f} minutes")
print(f"Checkpoint: {resnet50_checkpoint_path}")

Epoch 01/20 | Train Loss 0.4243 | Val Loss 0.3675 | Val Acc 0.8283 | Val F1 0.5312 | Val AUC 0.9101 | Time 66.1s
  Best checkpoint saved.
Epoch 02/20 | Train Loss 0.2904 | Val Loss 0.3047 | Val Acc 0.8760 | Val F1 0.5778 | Val AUC 0.9106 | Time 65.4s
  Best checkpoint saved.
Epoch 03/20 | Train Loss 0.2231 | Val Loss 0.3199 | Val Acc 0.8584 | Val F1 0.5651 | Val AUC 0.9179 | Time 65.4s
Epoch 04/20 | Train Loss 0.1751 | Val Loss 0.2860 | Val Acc 0.8995 | Val F1 0.6031 | Val AUC 0.9091 | Time 65.7s
  Best checkpoint saved.
Epoch 05/20 | Train Loss 0.1600 | Val Loss 0.3044 | Val Acc 0.8753 | Val F1 0.5802 | Val AUC 0.9201 | Time 65.4s
Epoch 06/20 | Train Loss 0.1264 | Val Loss 0.3689 | Val Acc 0.8982 | Val F1 0.5642 | Val AUC 0.8659 | Time 65.2s
Epoch 07/20 | Train Loss 0.1143 | Val Loss 0.3036 | Val Acc 0.8969 | Val F1 0.6256 | Val AUC 0.9280 | Time 64.8s
  Best checkpoint saved.
Epoch 08/20 | Train Loss 0.0931 | Val Loss 0.3753 | Val Acc 0.8779 | Val F1 0.5760 | Val AUC 0.9193 | Time 64

In [75]:
# Evaluate the best ResNet50 checkpoint on the untouched HAM10000 test set

criterion = nn.CrossEntropyLoss()

resnet50_checkpoint = "/kaggle/working/ham10000_resnet50_balanced_best.pth"

checkpoint = torch.load(
    resnet50_checkpoint,
    map_location=device,
    weights_only=False
)

resnet50_model.load_state_dict(checkpoint["model_state_dict"])

print(f"Loaded best checkpoint from epoch: {checkpoint['epoch']}")
print(f"Best validation F1: {checkpoint['best_val_f1']:.4f}")

resnet50_test_results = evaluate_model(
    resnet50_model,
    test_loader,
    criterion,
    device
)

print("\n" + "=" * 60)
print("HAM10000 RESNET50 OFFICIAL TEST RESULTS")
print("=" * 60)

for key, value in resnet50_test_results.items():
    if key != "confusion_matrix":
        print(f"{key}: {value:.6f}")

print("\nConfusion Matrix:")
print(resnet50_test_results["confusion_matrix"])

Loaded best checkpoint from epoch: 7
Best validation F1: 0.6256

HAM10000 RESNET50 OFFICIAL TEST RESULTS
loss: 0.344010
accuracy: 0.881491
precision: 0.477178
recall: 0.688623
f1: 0.563725
roc_auc: 0.908995

Confusion Matrix:
[[1209  126]
 [  52  115]]


In [77]:
# Save the official ResNet50 results and complete training history

import json
import torch

resnet50_results_path = "/kaggle/working/ham10000_resnet50_balanced_results.json"
resnet50_history_path = "/kaggle/working/ham10000_resnet50_balanced_training_history.pt"

# Reconstruct the recorded training history
resnet50_training_history = {
    "train_loss": [
        0.4243, 0.2904, 0.2231, 0.1751,
        0.1600, 0.1264, 0.1143, 0.0931,
        0.0798, 0.0774, 0.0499, 0.0435
    ],
    "val_loss": [
        0.3675, 0.3047, 0.3199, 0.2860,
        0.3044, 0.3689, 0.3036, 0.3753,
        0.3463, 0.4919, 0.3961, 0.3636
    ],
    "val_accuracy": [
        0.8283, 0.8760, 0.8584, 0.8995,
        0.8753, 0.8982, 0.8969, 0.8779,
        0.9080, 0.8956, 0.9047, 0.9210
    ],
    "val_f1": [
        0.5312, 0.5778, 0.5651, 0.6031,
        0.5802, 0.5642, 0.6256, 0.5760,
        0.6199, 0.5152, 0.6033, 0.6254
    ],
    "val_auc": [
        0.9101, 0.9106, 0.9179, 0.9091,
        0.9201, 0.8659, 0.9280, 0.9193,
        0.9117, 0.8649, 0.9078, 0.8968
    ]
}

# Save official test results
resnet50_results_to_save = {
    "loss": 0.344010,
    "accuracy": 0.881491,
    "precision": 0.477178,
    "recall": 0.688623,
    "f1": 0.563725,
    "roc_auc": 0.908995,
    "confusion_matrix": [[1209, 126], [52, 115]],
    "best_epoch": 7,
    "best_val_f1": 0.6256
}

with open(resnet50_results_path, "w") as f:
    json.dump(resnet50_results_to_save, f, indent=4)

# Save training history
torch.save(
    {
        "history": resnet50_training_history,
        "best_epoch": 7,
        "best_val_f1": 0.6256
    },
    resnet50_history_path
)

print("ResNet50 artifacts saved successfully.")
print(f"Results: {resnet50_results_path}")
print(f"History: {resnet50_history_path}")

ResNet50 artifacts saved successfully.
Results: /kaggle/working/ham10000_resnet50_balanced_results.json
History: /kaggle/working/ham10000_resnet50_balanced_training_history.pt


In [78]:
# Verify the ResNet50 model artifacts before creating the backup

import os

resnet50_files = [
    "/kaggle/working/ham10000_resnet50_balanced_best.pth",
    "/kaggle/working/ham10000_resnet50_balanced_training_history.pt",
    "/kaggle/working/ham10000_resnet50_balanced_results.json"
]

print("=" * 60)
print("RESNET50 ARTIFACT VERIFICATION")
print("=" * 60)

for file_path in resnet50_files:
    if os.path.exists(file_path):
        size_mb = os.path.getsize(file_path) / (1024 ** 2)
        print(f"✓ {os.path.basename(file_path)}")
        print(f"  Size: {size_mb:.2f} MB")
    else:
        print(f"✗ MISSING: {file_path}")

RESNET50 ARTIFACT VERIFICATION
✓ ham10000_resnet50_balanced_best.pth
  Size: 269.53 MB
✓ ham10000_resnet50_balanced_training_history.pt
  Size: 0.00 MB
✓ ham10000_resnet50_balanced_results.json
  Size: 0.00 MB


In [79]:
# Create a complete ResNet50 backup ZIP

import zipfile
import os

backup_path = "/kaggle/working/HAM10000_ResNet50_Balanced_Baseline_Backup.zip"

resnet50_files = [
    "/kaggle/working/ham10000_resnet50_balanced_best.pth",
    "/kaggle/working/ham10000_resnet50_balanced_training_history.pt",
    "/kaggle/working/ham10000_resnet50_balanced_results.json"
]

with zipfile.ZipFile(
    backup_path,
    "w",
    compression=zipfile.ZIP_STORED
) as zipf:

    for file_path in resnet50_files:
        zipf.write(
            file_path,
            arcname=os.path.basename(file_path)
        )

backup_size_mb = os.path.getsize(backup_path) / (1024 ** 2)

print("=" * 60)
print("RESNET50 BACKUP CREATED")
print("=" * 60)
print(f"Backup: {backup_path}")
print(f"Size: {backup_size_mb:.2f} MB")

with zipfile.ZipFile(backup_path, "r") as zipf:
    print("\nBackup contents:")
    for name in zipf.namelist():
        print(f"✓ {name}")

RESNET50 BACKUP CREATED
Backup: /kaggle/working/HAM10000_ResNet50_Balanced_Baseline_Backup.zip
Size: 269.53 MB

Backup contents:
✓ ham10000_resnet50_balanced_best.pth
✓ ham10000_resnet50_balanced_training_history.pt
✓ ham10000_resnet50_balanced_results.json


In [81]:
# Initialize the ImageNet-pretrained EfficientNet-B0 classifier

from torchvision import models
import torch.nn as nn

efficientnet_model = models.efficientnet_b0(
    weights=models.EfficientNet_B0_Weights.DEFAULT
)

efficientnet_model.classifier[1] = nn.Linear(
    efficientnet_model.classifier[1].in_features,
    NUM_CLASSES
)

efficientnet_model = efficientnet_model.to(device)

if torch.cuda.device_count() > 1:
    efficientnet_model = nn.DataParallel(efficientnet_model)

print("EfficientNet-B0 initialized successfully.")
print(f"Parameters: {sum(p.numel() for p in efficientnet_model.parameters()):,}")
print(f"Device: {device}")
print(f"GPU count: {torch.cuda.device_count()}")

EfficientNet-B0 initialized successfully.
Parameters: 4,010,110
Device: cuda
GPU count: 2


In [83]:
# Configure EfficientNet-B0 training with the established benchmark settings

import torch.optim as optim

efficientnet_criterion = nn.CrossEntropyLoss()

efficientnet_optimizer = optim.AdamW(
    efficientnet_model.parameters(),
    lr=1e-4,
    weight_decay=1e-5
)

efficientnet_scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    efficientnet_optimizer,
    mode="max",
    factor=0.5,
    patience=2
)

MAX_EPOCHS = 20
EARLY_STOPPING_PATIENCE = 5

print("=" * 60)
print("EFFICIENTNET-B0 TRAINING CONFIGURATION")
print("=" * 60)
print("Loss: CrossEntropyLoss")
print("Optimizer: AdamW")
print("Learning rate: 0.0001")
print("Weight decay: 0.00001")
print("Scheduler: ReduceLROnPlateau")
print("Max epochs: 20")
print("Early stopping patience: 5")
print("Training loader: balanced_train_loader")
print("Validation loader: val_loader")
print("Test loader: test_loader")
print("Model selection: Validation F1")

EFFICIENTNET-B0 TRAINING CONFIGURATION
Loss: CrossEntropyLoss
Optimizer: AdamW
Learning rate: 0.0001
Weight decay: 0.00001
Scheduler: ReduceLROnPlateau
Max epochs: 20
Early stopping patience: 5
Training loader: balanced_train_loader
Validation loader: val_loader
Test loader: test_loader
Model selection: Validation F1


In [84]:
# Train the EfficientNet-B0 benchmark model

import time

efficientnet_best_val_f1 = -1.0
efficientnet_best_epoch = 0
efficientnet_epochs_without_improvement = 0

efficientnet_training_history = {
    "train_loss": [],
    "val_loss": [],
    "val_accuracy": [],
    "val_precision": [],
    "val_recall": [],
    "val_f1": [],
    "val_auc": []
}

efficientnet_checkpoint_path = (
    "/kaggle/working/ham10000_efficientnet_b0_balanced_best.pth"
)

training_start_time = time.time()

for epoch in range(1, MAX_EPOCHS + 1):

    epoch_start_time = time.time()

    efficientnet_model.train()

    running_loss = 0.0
    total_samples = 0

    for images, labels in balanced_train_loader:

        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        efficientnet_optimizer.zero_grad(set_to_none=True)

        outputs = efficientnet_model(images)
        loss = efficientnet_criterion(outputs, labels)

        loss.backward()
        efficientnet_optimizer.step()

        batch_size = images.size(0)
        running_loss += loss.item() * batch_size
        total_samples += batch_size

    train_loss = running_loss / total_samples

    val_results = evaluate_model(
        efficientnet_model,
        val_loader,
        efficientnet_criterion,
        device
    )

    val_loss = val_results["loss"]
    val_accuracy = val_results["accuracy"]
    val_precision = val_results["precision"]
    val_recall = val_results["recall"]
    val_f1 = val_results["f1"]
    val_auc = val_results["roc_auc"]

    efficientnet_training_history["train_loss"].append(train_loss)
    efficientnet_training_history["val_loss"].append(val_loss)
    efficientnet_training_history["val_accuracy"].append(val_accuracy)
    efficientnet_training_history["val_precision"].append(val_precision)
    efficientnet_training_history["val_recall"].append(val_recall)
    efficientnet_training_history["val_f1"].append(val_f1)
    efficientnet_training_history["val_auc"].append(val_auc)

    efficientnet_scheduler.step(val_f1)

    epoch_time = time.time() - epoch_start_time

    print(
        f"Epoch {epoch:02d}/{MAX_EPOCHS} | "
        f"Train Loss {train_loss:.4f} | "
        f"Val Loss {val_loss:.4f} | "
        f"Val Acc {val_accuracy:.4f} | "
        f"Val F1 {val_f1:.4f} | "
        f"Val AUC {val_auc:.4f} | "
        f"Time {epoch_time:.1f}s"
    )

    if val_f1 > efficientnet_best_val_f1:

        efficientnet_best_val_f1 = val_f1
        efficientnet_best_epoch = epoch
        efficientnet_epochs_without_improvement = 0

        torch.save(
            {
                "epoch": epoch,
                "model_state_dict": efficientnet_model.state_dict(),
                "optimizer_state_dict": efficientnet_optimizer.state_dict(),
                "scheduler_state_dict": efficientnet_scheduler.state_dict(),
                "best_val_f1": efficientnet_best_val_f1
            },
            efficientnet_checkpoint_path
        )

        print("  Best checkpoint saved.")

    else:
        efficientnet_epochs_without_improvement += 1

    if efficientnet_epochs_without_improvement >= EARLY_STOPPING_PATIENCE:
        print(
            f"Early stopping triggered after "
            f"{EARLY_STOPPING_PATIENCE} epochs without improvement."
        )
        break

total_training_time = time.time() - training_start_time

print("\nEfficientNet-B0 training completed.")
print(f"Best epoch: {efficientnet_best_epoch}")
print(f"Best validation F1: {efficientnet_best_val_f1:.4f}")
print(f"Total training time: {total_training_time / 60:.2f} minutes")
print(f"Checkpoint: {efficientnet_checkpoint_path}")

Epoch 01/20 | Train Loss 0.4562 | Val Loss 0.3981 | Val Acc 0.7983 | Val F1 0.4909 | Val AUC 0.9176 | Time 71.0s
  Best checkpoint saved.
Epoch 02/20 | Train Loss 0.3393 | Val Loss 0.2937 | Val Acc 0.8695 | Val F1 0.5798 | Val AUC 0.9172 | Time 70.3s
  Best checkpoint saved.
Epoch 03/20 | Train Loss 0.2760 | Val Loss 0.2608 | Val Acc 0.8890 | Val F1 0.6028 | Val AUC 0.9187 | Time 70.8s
  Best checkpoint saved.
Epoch 04/20 | Train Loss 0.2270 | Val Loss 0.2551 | Val Acc 0.8969 | Val F1 0.6127 | Val AUC 0.9205 | Time 70.8s
  Best checkpoint saved.
Epoch 05/20 | Train Loss 0.1914 | Val Loss 0.2925 | Val Acc 0.8799 | Val F1 0.5799 | Val AUC 0.9144 | Time 70.3s
Epoch 06/20 | Train Loss 0.1634 | Val Loss 0.2780 | Val Acc 0.8923 | Val F1 0.6024 | Val AUC 0.9194 | Time 70.5s
Epoch 07/20 | Train Loss 0.1333 | Val Loss 0.2670 | Val Acc 0.9138 | Val F1 0.5951 | Val AUC 0.9017 | Time 70.4s
Epoch 08/20 | Train Loss 0.1143 | Val Loss 0.2694 | Val Acc 0.9021 | Val F1 0.5989 | Val AUC 0.9201 | Time 69

In [85]:
# Evaluate the best EfficientNet-B0 checkpoint on the untouched HAM10000 test set

efficientnet_checkpoint_path = (
    "/kaggle/working/ham10000_efficientnet_b0_balanced_best.pth"
)

efficientnet_checkpoint = torch.load(
    efficientnet_checkpoint_path,
    map_location=device,
    weights_only=False
)

efficientnet_model.load_state_dict(
    efficientnet_checkpoint["model_state_dict"]
)

print(
    f"Loaded best checkpoint from epoch: "
    f"{efficientnet_checkpoint['epoch']}"
)

print(
    f"Best validation F1: "
    f"{efficientnet_checkpoint['best_val_f1']:.4f}"
)

efficientnet_test_results = evaluate_model(
    efficientnet_model,
    test_loader,
    efficientnet_criterion,
    device
)

print("\n" + "=" * 60)
print("HAM10000 EFFICIENTNET-B0 OFFICIAL TEST RESULTS")
print("=" * 60)

for key, value in efficientnet_test_results.items():
    if key != "confusion_matrix":
        print(f"{key}: {value:.6f}")

print("\nConfusion Matrix:")
print(efficientnet_test_results["confusion_matrix"])

Loaded best checkpoint from epoch: 9
Best validation F1: 0.6220

HAM10000 EFFICIENTNET-B0 OFFICIAL TEST RESULTS
loss: 0.332711
accuracy: 0.895473
precision: 0.534722
recall: 0.461078
f1: 0.495177
roc_auc: 0.888434

Confusion Matrix:
[[1268   67]
 [  90   77]]


In [86]:
# Save EfficientNet-B0 test results and training history for reproducibility

import json
import torch

efficientnet_results_path = (
    "/kaggle/working/ham10000_efficientnet_b0_balanced_results.json"
)

efficientnet_history_path = (
    "/kaggle/working/ham10000_efficientnet_b0_balanced_training_history.pt"
)

with open(efficientnet_results_path, "w") as f:
    json.dump(
        {
            key: value.tolist() if hasattr(value, "tolist") else value
            for key, value in efficientnet_test_results.items()
        },
        f,
        indent=4
    )

torch.save(
    efficientnet_training_history,
    efficientnet_history_path
)

print("EfficientNet-B0 results saved:")
print(efficientnet_results_path)

print("\nEfficientNet-B0 training history saved:")
print(efficientnet_history_path)

EfficientNet-B0 results saved:
/kaggle/working/ham10000_efficientnet_b0_balanced_results.json

EfficientNet-B0 training history saved:
/kaggle/working/ham10000_efficientnet_b0_balanced_training_history.pt


In [87]:
# Verify all EfficientNet-B0 research artifacts before creating the backup

import os

efficientnet_files = [
    "/kaggle/working/ham10000_efficientnet_b0_balanced_best.pth",
    "/kaggle/working/ham10000_efficientnet_b0_balanced_training_history.pt",
    "/kaggle/working/ham10000_efficientnet_b0_balanced_results.json"
]

print("=" * 60)
print("EFFICIENTNET-B0 ARTIFACT VERIFICATION")
print("=" * 60)

all_present = True

for file_path in efficientnet_files:
    exists = os.path.exists(file_path)
    all_present = all_present and exists

    if exists:
        size_mb = os.path.getsize(file_path) / (1024 ** 2)
        print(f"✓ {os.path.basename(file_path)} — {size_mb:.2f} MB")
    else:
        print(f"✗ MISSING: {file_path}")

print("\n" + "=" * 60)

if all_present:
    print("ALL 3 ARTIFACTS VERIFIED")
else:
    print("WARNING: ONE OR MORE ARTIFACTS ARE MISSING")

EFFICIENTNET-B0 ARTIFACT VERIFICATION
✓ ham10000_efficientnet_b0_balanced_best.pth — 46.38 MB
✓ ham10000_efficientnet_b0_balanced_training_history.pt — 0.00 MB
✓ ham10000_efficientnet_b0_balanced_results.json — 0.00 MB

ALL 3 ARTIFACTS VERIFIED


In [88]:
# Create a complete EfficientNet-B0 backup archive

import zipfile
import os

efficientnet_backup_path = (
    "/kaggle/working/HAM10000_EfficientNetB0_Balanced_Baseline_Backup.zip"
)

efficientnet_files = [
    "/kaggle/working/ham10000_efficientnet_b0_balanced_best.pth",
    "/kaggle/working/ham10000_efficientnet_b0_balanced_training_history.pt",
    "/kaggle/working/ham10000_efficientnet_b0_balanced_results.json"
]

with zipfile.ZipFile(
    efficientnet_backup_path,
    "w",
    compression=zipfile.ZIP_STORED
) as zipf:

    for file_path in efficientnet_files:
        zipf.write(
            file_path,
            arcname=os.path.basename(file_path)
        )

backup_size_mb = os.path.getsize(efficientnet_backup_path) / (1024 ** 2)

print("=" * 60)
print("EFFICIENTNET-B0 BACKUP CREATED")
print("=" * 60)
print(f"Backup: {efficientnet_backup_path}")
print(f"Size: {backup_size_mb:.2f} MB")

EFFICIENTNET-B0 BACKUP CREATED
Backup: /kaggle/working/HAM10000_EfficientNetB0_Balanced_Baseline_Backup.zip
Size: 46.39 MB


In [89]:
# Verify the EfficientNet-B0 backup contains all required research artifacts

import zipfile
import os

efficientnet_backup_path = (
    "/kaggle/working/HAM10000_EfficientNetB0_Balanced_Baseline_Backup.zip"
)

required_files = {
    "ham10000_efficientnet_b0_balanced_best.pth",
    "ham10000_efficientnet_b0_balanced_training_history.pt",
    "ham10000_efficientnet_b0_balanced_results.json"
}

with zipfile.ZipFile(efficientnet_backup_path, "r") as zipf:
    contents = set(zipf.namelist())

print("=" * 60)
print("EFFICIENTNET-B0 BACKUP VERIFICATION")
print("=" * 60)

for filename in sorted(required_files):
    if filename in contents:
        info = zipf.getinfo(filename)
        size_mb = info.file_size / (1024 ** 2)
        print(f"✓ {filename} — {size_mb:.2f} MB")
    else:
        print(f"✗ MISSING: {filename}")

print("\n" + "=" * 60)

if required_files.issubset(contents):
    print("BACKUP VERIFIED — ALL 3 ARTIFACTS PRESENT")
else:
    print("WARNING — BACKUP IS INCOMPLETE")

EFFICIENTNET-B0 BACKUP VERIFICATION
✓ ham10000_efficientnet_b0_balanced_best.pth — 46.38 MB
✓ ham10000_efficientnet_b0_balanced_results.json — 0.00 MB
✓ ham10000_efficientnet_b0_balanced_training_history.pt — 0.00 MB

BACKUP VERIFIED — ALL 3 ARTIFACTS PRESENT


In [90]:
# Check for existing VGG19 result variables and saved artifacts

import os

print("=" * 60)
print("VGG19 EXISTING RESULTS CHECK")
print("=" * 60)

# Check common VGG19 variables currently held in memory
vgg19_variables = [
    "vgg19_test_results",
    "vgg19_training_history",
    "vgg19_checkpoint"
]

for variable_name in vgg19_variables:
    print(
        f"{variable_name}: "
        f"{'AVAILABLE' if variable_name in globals() else 'NOT AVAILABLE'}"
    )

print("\nSaved VGG19 files in /kaggle/working:")

vgg19_files = []

for root, dirs, files in os.walk("/kaggle/working"):
    for filename in files:
        if "vgg19" in filename.lower():
            vgg19_files.append(os.path.join(root, filename))

if vgg19_files:
    for path in sorted(vgg19_files):
        size_mb = os.path.getsize(path) / (1024 ** 2)
        print(f"✓ {path} — {size_mb:.2f} MB")
else:
    print("No VGG19 files found.")


VGG19 EXISTING RESULTS CHECK
vgg19_test_results: NOT AVAILABLE
vgg19_training_history: NOT AVAILABLE
vgg19_checkpoint: NOT AVAILABLE

Saved VGG19 files in /kaggle/working:
✓ /kaggle/working/HAM10000_VGG19_Balanced_Baseline_Backup.zip — 1518.43 MB
✓ /kaggle/working/ham10000_vgg19_balanced_best.pth — 1597.40 MB
✓ /kaggle/working/ham10000_vgg19_balanced_results.json — 0.00 MB
✓ /kaggle/working/ham10000_vgg19_balanced_training_history.pt — 0.00 MB


In [91]:
# Read the saved VGG19 official test results without retraining the model

import json

vgg19_results_path = (
    "/kaggle/working/ham10000_vgg19_balanced_results.json"
)

with open(vgg19_results_path, "r") as f:
    vgg19_saved_results = json.load(f)

print("=" * 60)
print("HAM10000 VGG19 SAVED TEST RESULTS")
print("=" * 60)

for key, value in vgg19_saved_results.items():
    print(f"{key}: {value}")

print("=" * 60)

HAM10000 VGG19 SAVED TEST RESULTS
model: VGG19
dataset: HAM10000
task: Binary melanoma vs non-melanoma classification
best_epoch: 2
best_validation_f1: 0.6440677966101694
test_loss: 0.2695222770207414
test_accuracy: 0.903462050599201
test_precision: 0.56875
test_recall: 0.5449101796407185
test_f1: 0.5565749235474006
test_roc_auc: 0.8724842449931597
confusion_matrix: [[1266, 69], [76, 91]]
seed: 42
batch_size: 32
learning_rate: 0.0001
weight_decay: 1e-05
max_epochs: 20
early_stopping_patience: 5
training_strategy: WeightedRandomSampler with ordinary CrossEntropyLoss


In [23]:
# --- Verify the fresh Kaggle runtime and GPU ---

import platform
import subprocess

print("Python version:", platform.python_version())
print("Operating system:", platform.system())

print("\nGPU information:")

result = subprocess.run(
    [
        "nvidia-smi",
        "--query-gpu=name,memory.total,driver_version",
        "--format=csv,noheader"
    ],
    capture_output=True,
    text=True
)

if result.returncode == 0:
    print(result.stdout)
else:
    print("GPU check failed.")
    print(result.stderr)

Python version: 3.12.13
Operating system: Linux

GPU information:
Tesla T4, 15360 MiB, 580.159.04
Tesla T4, 15360 MiB, 580.159.04



In [22]:
# --- Configure reproducibility for the research experiments ---

import os
import random
import numpy as np
import torch

SEED = 42

os.environ["PYTHONHASHSEED"] = str(SEED)

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

print("Random seed:", SEED)
print("PyTorch seed configured")
print("CUDA seed configured:", torch.cuda.is_available())
print("Deterministic mode:", torch.backends.cudnn.deterministic)
print("Benchmark mode:", torch.backends.cudnn.benchmark)

Random seed: 42
PyTorch seed configured
CUDA seed configured: True
Deterministic mode: True
Benchmark mode: False


In [23]:
# --- Verify PyTorch and CUDA availability ---

import torch

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("CUDA version:", torch.version.cuda)
    print("GPU count:", torch.cuda.device_count())

    for i in range(torch.cuda.device_count()):
        print(f"GPU {i}:", torch.cuda.get_device_name(i))
        print(f"GPU {i} memory:", round(torch.cuda.get_device_properties(i).total_memory / (1024 ** 3), 2), "GB")

    device = torch.device("cuda")
    print("Selected device:", device)
else:
    device = torch.device("cpu")
    print("Selected device:", device)

PyTorch version: 2.10.0+cu128
CUDA available: True
CUDA version: 12.8
GPU count: 2
GPU 0: Tesla T4
GPU 0 memory: 14.56 GB
GPU 1: Tesla T4
GPU 1 memory: 14.56 GB
Selected device: cuda


In [25]:
# --- Locate and inspect the attached HAM10000 Kaggle dataset ---

from pathlib import Path

INPUT_ROOT = Path("/kaggle/input")

print("Kaggle input directory:")
print(INPUT_ROOT)

print("\nAttached datasets:")

for item in INPUT_ROOT.iterdir():
    print(" -", item.name)

Kaggle input directory:
/kaggle/input

Attached datasets:
 - datasets


In [27]:
# --- Inspect the attached Kaggle dataset directory ---

from pathlib import Path

DATASETS_ROOT = Path("/kaggle/input/datasets")

print("Contents of /kaggle/input/datasets:\n")

for item in sorted(DATASETS_ROOT.iterdir()):
    if item.is_dir():
        print("DIRECTORY:", item.name)
    else:
        print("FILE:", item.name)

Contents of /kaggle/input/datasets:

DIRECTORY: akansha1009


In [28]:
# --- Identify the exact HAM10000 dataset folder ---

from pathlib import Path

DATASETS_ROOT = Path("/kaggle/input/datasets/akansha1009")

print("Contents of the attached dataset namespace:\n")

for item in sorted(DATASETS_ROOT.iterdir()):
    if item.is_dir():
        print("DIRECTORY:", item.name)
    else:
        print("FILE:", item.name)

Contents of the attached dataset namespace:

DIRECTORY: ham10000-preprocessed-segmentation


In [30]:
# --- Inspect the attached HAM10000 dataset structure ---

from pathlib import Path

HAM_ROOT = Path(
    "/kaggle/input/datasets/akansha1009/ham10000-preprocessed-segmentation"
)

print("HAM10000 dataset root:")
print(HAM_ROOT)

print("\nTop-level contents:")

for item in sorted(HAM_ROOT.iterdir()):
    if item.is_dir():
        print("DIRECTORY:", item.name)
    else:
        print("FILE:", item.name)

HAM10000 dataset root:
/kaggle/input/datasets/akansha1009/ham10000-preprocessed-segmentation

Top-level contents:
DIRECTORY: test
DIRECTORY: train
DIRECTORY: validation


In [31]:
# --- Inspect HAM10000 train, validation, and test contents ---

for split_name in ["train", "validation", "test"]:
    split_path = HAM_ROOT / split_name

    print(f"\n{'-' * 60}")
    print(f"{split_name.upper()}")
    print(f"{'-' * 60}")

    items = sorted(split_path.iterdir())

    for item in items[:20]:
        if item.is_dir():
            print("DIRECTORY:", item.name)
        else:
            print("FILE:", item.name)

    if len(items) > 20:
        print(f"... and {len(items) - 20} more items")

    print("Total top-level items:", len(items))


------------------------------------------------------------
TRAIN
------------------------------------------------------------
FILE: images.npy
FILE: masks.npy
FILE: metadata.csv
Total top-level items: 3

------------------------------------------------------------
VALIDATION
------------------------------------------------------------
FILE: images.npy
FILE: masks.npy
FILE: metadata.csv
Total top-level items: 3

------------------------------------------------------------
TEST
------------------------------------------------------------
FILE: images.npy
FILE: masks.npy
FILE: metadata.csv
Total top-level items: 3


In [33]:
# --- Inspect HAM10000 metadata and diagnostic columns ---

import pandas as pd

for split_name in ["train", "validation", "test"]:
    metadata_path = HAM_ROOT / split_name / "metadata.csv"
    df = pd.read_csv(metadata_path)

    print(f"\n{'-' * 60}")
    print(f"{split_name.upper()} METADATA")
    print(f"{'-' * 60}")

    print("Shape:", df.shape)
    print("Columns:")
    print(df.columns.tolist())

    print("\nFirst 3 rows:")
    display(df.head(3))


------------------------------------------------------------
TRAIN METADATA
------------------------------------------------------------
Shape: (6981, 2)
Columns:
['image_id', 'dx']

First 3 rows:


,image_id,dx
0,ISIC_0027419,bkl
1,ISIC_0025030,bkl
2,ISIC_0026769,bkl



------------------------------------------------------------
VALIDATION METADATA
------------------------------------------------------------
Shape: (1532, 2)
Columns:
['image_id', 'dx']

First 3 rows:


,image_id,dx
0,ISIC_0025767,bkl
1,ISIC_0032417,bkl
2,ISIC_0025915,bkl



------------------------------------------------------------
TEST METADATA
------------------------------------------------------------
Shape: (1502, 2)
Columns:
['image_id', 'dx']

First 3 rows:


,image_id,dx
0,ISIC_0029176,bkl
1,ISIC_0029068,bkl
2,ISIC_0025837,bkl


In [34]:
# --- Verify all HAM10000 diagnostic categories and split counts ---

for split_name in ["train", "validation", "test"]:
    metadata_path = HAM_ROOT / split_name / "metadata.csv"
    df = pd.read_csv(metadata_path)

    print(f"\n{'-' * 60}")
    print(f"{split_name.upper()}")
    print(f"{'-' * 60}")

    print("Total images:", len(df))
    print("\nDiagnostic categories:")
    print(df["dx"].value_counts().sort_index())


------------------------------------------------------------
TRAIN
------------------------------------------------------------
Total images: 6981

Diagnostic categories:
dx
akiec     222
bcc       361
bkl       772
df         71
mel       773
nv       4683
vasc       99
Name: count, dtype: int64

------------------------------------------------------------
VALIDATION
------------------------------------------------------------
Total images: 1532

Diagnostic categories:
dx
akiec      53
bcc        82
bkl       160
df         24
mel       173
nv       1018
vasc       22
Name: count, dtype: int64

------------------------------------------------------------
TEST
------------------------------------------------------------
Total images: 1502

Diagnostic categories:
dx
akiec      52
bcc        71
bkl       167
df         20
mel       167
nv       1004
vasc       21
Name: count, dtype: int64


In [35]:
# --- Freeze HAM10000 binary melanoma classification labels ---

BINARY_LABEL_MAP = {
    "mel": 1,
    "akiec": 0,
    "bcc": 0,
    "bkl": 0,
    "df": 0,
    "nv": 0,
    "vasc": 0,
}

print("Binary label mapping:")
print("1 -> Melanoma")
print("0 -> Non-Melanoma")

print("\nBinary class distribution:")

for split_name in ["train", "validation", "test"]:
    metadata_path = HAM_ROOT / split_name / "metadata.csv"
    df = pd.read_csv(metadata_path)

    df["binary_label"] = df["dx"].map(BINARY_LABEL_MAP)

    print(f"\n{'-' * 50}")
    print(split_name.upper())
    print(f"{'-' * 50}")

    print("Total:", len(df))
    print("Melanoma:", (df["binary_label"] == 1).sum())
    print("Non-Melanoma:", (df["binary_label"] == 0).sum())
    print("Missing labels:", df["binary_label"].isna().sum())

Binary label mapping:
1 -> Melanoma
0 -> Non-Melanoma

Binary class distribution:

--------------------------------------------------
TRAIN
--------------------------------------------------
Total: 6981
Melanoma: 773
Non-Melanoma: 6208
Missing labels: 0

--------------------------------------------------
VALIDATION
--------------------------------------------------
Total: 1532
Melanoma: 173
Non-Melanoma: 1359
Missing labels: 0

--------------------------------------------------
TEST
--------------------------------------------------
Total: 1502
Melanoma: 167
Non-Melanoma: 1335
Missing labels: 0


In [36]:
# --- Verify HAM10000 image, mask, and metadata array alignment ---

import numpy as np

for split_name in ["train", "validation", "test"]:
    split_path = HAM_ROOT / split_name

    images = np.load(split_path / "images.npy", mmap_mode="r")
    masks = np.load(split_path / "masks.npy", mmap_mode="r")
    metadata = pd.read_csv(split_path / "metadata.csv")

    print(f"\n{'-' * 60}")
    print(split_name.upper())
    print(f"{'-' * 60}")

    print("Images shape:", images.shape)
    print("Images dtype:", images.dtype)

    print("Masks shape:", masks.shape)
    print("Masks dtype:", masks.dtype)

    print("Metadata shape:", metadata.shape)

    print("Image count matches metadata:", len(images) == len(metadata))
    print("Mask count matches metadata:", len(masks) == len(metadata))


------------------------------------------------------------
TRAIN
------------------------------------------------------------
Images shape: (6981, 224, 224, 3)
Images dtype: float32
Masks shape: (6981, 224, 224)
Masks dtype: uint8
Metadata shape: (6981, 2)
Image count matches metadata: True
Mask count matches metadata: True

------------------------------------------------------------
VALIDATION
------------------------------------------------------------
Images shape: (1532, 224, 224, 3)
Images dtype: float32
Masks shape: (1532, 224, 224)
Masks dtype: uint8
Metadata shape: (1532, 2)
Image count matches metadata: True
Mask count matches metadata: True

------------------------------------------------------------
TEST
------------------------------------------------------------
Images shape: (1502, 224, 224, 3)
Images dtype: float32
Masks shape: (1502, 224, 224)
Masks dtype: uint8
Metadata shape: (1502, 2)
Image count matches metadata: True
Mask count matches metadata: True


In [37]:
# --- Inspect HAM10000 image value ranges ---

for split_name in ["train", "validation", "test"]:
    images = np.load(
        HAM_ROOT / split_name / "images.npy",
        mmap_mode="r"
    )

    print(f"\n{'-' * 50}")
    print(split_name.upper())
    print(f"{'-' * 50}")

    print("Minimum:", float(images.min()))
    print("Maximum:", float(images.max()))
    print("Mean:", float(images.mean()))
    print("Standard deviation:", float(images.std()))


--------------------------------------------------
TRAIN
--------------------------------------------------
Minimum: 0.0
Maximum: 1.0
Mean: 0.6340610384941101
Standard deviation: 0.17963218688964844

--------------------------------------------------
VALIDATION
--------------------------------------------------
Minimum: 0.0
Maximum: 1.0
Mean: 0.6367025971412659
Standard deviation: 0.17934194207191467

--------------------------------------------------
TEST
--------------------------------------------------
Minimum: 0.0
Maximum: 1.0
Mean: 0.6325188875198364
Standard deviation: 0.17900870740413666


In [39]:
# --- Verify HAM10000 split identity and leakage ---

split_ids = {}

for split_name in ["train", "validation", "test"]:
    metadata_path = HAM_ROOT / split_name / "metadata.csv"
    df = pd.read_csv(metadata_path)

    ids = df["image_id"].astype(str)

    split_ids[split_name] = set(ids)

    print(f"\n{split_name.upper()}")
    print("Total IDs:", len(ids))
    print("Unique IDs:", ids.nunique())
    print("Duplicate IDs:", ids.duplicated().sum())

print("\nCross-split overlap:")

train_val = split_ids["train"] & split_ids["validation"]
train_test = split_ids["train"] & split_ids["test"]
val_test = split_ids["validation"] & split_ids["test"]

print("Train ∩ Validation:", len(train_val))
print("Train ∩ Test:", len(train_test))
print("Validation ∩ Test:", len(val_test))


TRAIN
Total IDs: 6981
Unique IDs: 6981
Duplicate IDs: 0

VALIDATION
Total IDs: 1532
Unique IDs: 1532
Duplicate IDs: 0

TEST
Total IDs: 1502
Unique IDs: 1502
Duplicate IDs: 0

Cross-split overlap:
Train ∩ Validation: 0
Train ∩ Test: 0
Validation ∩ Test: 0


In [40]:
# --- Build reusable HAM10000 classification Dataset ---

import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

class HAM10000ClassificationDataset(Dataset):
    def __init__(self, split_name, transform=None):
        self.split_name = split_name
        self.transform = transform

        split_path = HAM_ROOT / split_name

        self.images = np.load(
            split_path / "images.npy",
            mmap_mode="r"
        )

        self.metadata = pd.read_csv(
            split_path / "metadata.csv"
        )

        self.labels = (
            self.metadata["dx"]
            .map(BINARY_LABEL_MAP)
            .astype(np.int64)
            .values
        )

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, index):
        image = self.images[index]
        label = self.labels[index]

        image = torch.from_numpy(
            np.asarray(image).copy()
        ).permute(2, 0, 1).float()

        if self.transform is not None:
            image = self.transform(image)

        return image, torch.tensor(label, dtype=torch.long)


train_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.2),
    transforms.RandomRotation(15),
    transforms.ColorJitter(
        brightness=0.15,
        contrast=0.15,
        saturation=0.10,
        hue=0.02
    ),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

eval_transform = transforms.Compose([
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

train_dataset = HAM10000ClassificationDataset(
    "train",
    transform=train_transform
)

val_dataset = HAM10000ClassificationDataset(
    "validation",
    transform=eval_transform
)

test_dataset = HAM10000ClassificationDataset(
    "test",
    transform=eval_transform
)

print("Datasets created successfully.")
print("Train:", len(train_dataset))
print("Validation:", len(val_dataset))
print("Test:", len(test_dataset))

Datasets created successfully.
Train: 6981
Validation: 1532
Test: 1502


In [42]:
# --- Create HAM10000 DataLoaders and verify one training batch ---

BATCH_SIZE = 32

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

images, labels = next(iter(train_loader))

print("Train batches:", len(train_loader))
print("Validation batches:", len(val_loader))
print("Test batches:", len(test_loader))

print("\nOne training batch:")
print("Image shape:", images.shape)
print("Image dtype:", images.dtype)
print("Label shape:", labels.shape)
print("Label dtype:", labels.dtype)

print("\nBatch label values:", labels.unique().tolist())
print("Batch image minimum:", float(images.min()))
print("Batch image maximum:", float(images.max()))

Train batches: 219
Validation batches: 48
Test batches: 47

One training batch:
Image shape: torch.Size([32, 3, 224, 224])
Image dtype: torch.float32
Label shape: torch.Size([32])
Label dtype: torch.int64

Batch label values: [0, 1]
Batch image minimum: -2.1179039478302
Batch image maximum: 2.640000104904175


In [48]:
# --- Calculate class weights from the HAM10000 training split ---

train_metadata = pd.read_csv(
    HAM_ROOT / "train" / "metadata.csv"
)

train_labels = (
    train_metadata["dx"]
    .map(BINARY_LABEL_MAP)
    .astype(np.int64)
    .values
)

class_counts = np.bincount(train_labels)

class_weights = len(train_labels) / (
    len(class_counts) * class_counts
)

class_weights = torch.tensor(
    class_weights,
    dtype=torch.float32
)

print("Training class counts:")
print("Non-Melanoma:", class_counts[0])
print("Melanoma:", class_counts[1])

print("\nClass weights:")
print("Non-Melanoma:", class_weights[0].item())
print("Melanoma:", class_weights[1].item())

Training class counts:
Non-Melanoma: 6208
Melanoma: 773

Class weights:
Non-Melanoma: 0.5622583627700806
Melanoma: 4.515523910522461


In [51]:
# --- Define the common HAM10000 benchmark configuration ---

import time
import random

SEED = 42
NUM_CLASSES = 2
BATCH_SIZE = 32
NUM_WORKERS = 2

LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-5
NUM_EPOCHS = 20

EARLY_STOPPING_PATIENCE = 5

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Benchmark configuration")
print("-" * 50)
print("Seed:", SEED)
print("Number of classes:", NUM_CLASSES)
print("Batch size:", BATCH_SIZE)
print("Workers:", NUM_WORKERS)
print("Learning rate:", LEARNING_RATE)
print("Weight decay:", WEIGHT_DECAY)
print("Maximum epochs:", NUM_EPOCHS)
print("Early stopping patience:", EARLY_STOPPING_PATIENCE)
print("Device:", DEVICE)

if torch.cuda.is_available():
    print("GPU count:", torch.cuda.device_count())
    for i in range(torch.cuda.device_count()):
        print(f"GPU {i}:", torch.cuda.get_device_name(i))

Benchmark configuration
--------------------------------------------------
Seed: 42
Number of classes: 2
Batch size: 32
Workers: 2
Learning rate: 0.0001
Weight decay: 1e-05
Maximum epochs: 20
Early stopping patience: 5
Device: cuda
GPU count: 2
GPU 0: Tesla T4
GPU 1: Tesla T4


In [53]:
# --- Create the common binary classification evaluation function ---

import numpy as np
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix
)

def evaluate_model(model, loader, criterion, device):
    model.eval()

    total_loss = 0.0
    all_labels = []
    all_predictions = []
    all_probabilities = []

    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            outputs = model(images)
            loss = criterion(outputs, labels)

            probabilities = torch.softmax(outputs, dim=1)[:, 1]
            predictions = torch.argmax(outputs, dim=1)

            total_loss += loss.item() * images.size(0)

            all_labels.extend(labels.cpu().numpy())
            all_predictions.extend(predictions.cpu().numpy())
            all_probabilities.extend(probabilities.cpu().numpy())

    all_labels = np.array(all_labels)
    all_predictions = np.array(all_predictions)
    all_probabilities = np.array(all_probabilities)

    average_loss = total_loss / len(loader.dataset)

    accuracy = accuracy_score(
        all_labels,
        all_predictions
    )

    precision = precision_score(
        all_labels,
        all_predictions,
        zero_division=0
    )

    recall = recall_score(
        all_labels,
        all_predictions,
        zero_division=0
    )

    f1 = f1_score(
        all_labels,
        all_predictions,
        zero_division=0
    )

    roc_auc = roc_auc_score(
        all_labels,
        all_probabilities
    )

    cm = confusion_matrix(
        all_labels,
        all_predictions
    )

    return {
        "loss": average_loss,
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "roc_auc": roc_auc,
        "confusion_matrix": cm
    }

print("Evaluation function created successfully.")


Evaluation function created successfully.


In [45]:
# --- Define and prepare the HAM10000 AlexNet benchmark model ---

import torch
import torch.nn as nn
from torchvision import models

alexnet = models.alexnet(weights=models.AlexNet_Weights.DEFAULT)

alexnet.classifier[6] = nn.Linear(
    alexnet.classifier[6].in_features,
    NUM_CLASSES
)

alexnet = alexnet.to(DEVICE)

if torch.cuda.device_count() > 1:
    alexnet = nn.DataParallel(alexnet)

alexnet_parameter_count = sum(
    parameter.numel()
    for parameter in alexnet.parameters()
)

print("AlexNet model prepared successfully.")
print("-" * 50)
print("Parameters:", f"{alexnet_parameter_count:,}")
print("Device:", DEVICE)

if isinstance(alexnet, nn.DataParallel):
    print("Multi-GPU:", "Enabled")
    print("GPU count:", torch.cuda.device_count())
else:
    print("Multi-GPU:", "Disabled")

AlexNet model prepared successfully.
--------------------------------------------------
Parameters: 57,012,034
Device: cuda
Multi-GPU: Enabled
GPU count: 2


In [21]:
# --- Create the AlexNet loss function and optimizer ---

class_weights_tensor = torch.tensor(
    [0.5622583627700806, 4.515523910522461],
    dtype=torch.float32,
    device=DEVICE
)

alexnet_criterion = nn.CrossEntropyLoss(
    weight=class_weights_tensor
)

alexnet_optimizer = torch.optim.Adam(
    alexnet.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY
)

print("AlexNet training components created successfully.")
print("-" * 50)
print("Loss:", "Weighted CrossEntropyLoss")
print("Class weights:", class_weights_tensor.detach().cpu().numpy())
print("Optimizer:", "Adam")
print("Learning rate:", LEARNING_RATE)
print("Weight decay:", WEIGHT_DECAY)

AlexNet training components created successfully.
--------------------------------------------------
Loss: Weighted CrossEntropyLoss
Class weights: [0.56225836 4.515524  ]
Optimizer: Adam
Learning rate: 0.0001
Weight decay: 1e-05


In [46]:
# --- Train AlexNet on the frozen HAM10000 training split ---

from copy import deepcopy

alexnet_history = []

best_val_f1 = -1.0
best_epoch = 0
epochs_without_improvement = 0

alexnet_best_path = "/kaggle/working/ham10000_alexnet_best.pth"

training_start_time = time.time()

for epoch in range(1, NUM_EPOCHS + 1):
    epoch_start_time = time.time()

    alexnet.train()

    total_train_loss = 0.0
    train_samples = 0

    for images, labels in train_loader:
        images = images.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)

        alexnet_optimizer.zero_grad(set_to_none=True)

        outputs = alexnet(images)
        loss = alexnet_criterion(outputs, labels)

        loss.backward()
        alexnet_optimizer.step()

        total_train_loss += loss.item() * images.size(0)
        train_samples += images.size(0)

    train_loss = total_train_loss / train_samples

    val_metrics = evaluate_model(
        alexnet,
        val_loader,
        alexnet_criterion,
        DEVICE
    )

    epoch_time = time.time() - epoch_start_time

    epoch_record = {
        "epoch": epoch,
        "train_loss": train_loss,
        "val_loss": val_metrics["loss"],
        "val_accuracy": val_metrics["accuracy"],
        "val_precision": val_metrics["precision"],
        "val_recall": val_metrics["recall"],
        "val_f1": val_metrics["f1"],
        "val_roc_auc": val_metrics["roc_auc"],
        "epoch_time_seconds": epoch_time
    }

    alexnet_history.append(epoch_record)

    print(
        f"Epoch {epoch:02d}/{NUM_EPOCHS} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Val Loss: {val_metrics['loss']:.4f} | "
        f"Val Acc: {val_metrics['accuracy']:.4f} | "
        f"Val F1: {val_metrics['f1']:.4f} | "
        f"Val AUC: {val_metrics['roc_auc']:.4f} | "
        f"Time: {epoch_time:.1f}s"
    )

    if val_metrics["f1"] > best_val_f1:
        best_val_f1 = val_metrics["f1"]
        best_epoch = epoch
        epochs_without_improvement = 0

        torch.save(
            {
                "epoch": epoch,
                "model_state_dict": alexnet.state_dict(),
                "optimizer_state_dict": alexnet_optimizer.state_dict(),
                "best_val_f1": best_val_f1,
                "history": alexnet_history
            },
            alexnet_best_path
        )

        print("  Best validation F1 improved. Checkpoint saved.")
    else:
        epochs_without_improvement += 1
        print(
            f"  No improvement for "
            f"{epochs_without_improvement}/{EARLY_STOPPING_PATIENCE} epoch(s)."
        )

    if epochs_without_improvement >= EARLY_STOPPING_PATIENCE:
        print("Early stopping triggered.")
        break

training_time = time.time() - training_start_time

print("-" * 70)
print("AlexNet training completed.")
print("Best validation F1:", f"{best_val_f1:.4f}")
print("Best epoch:", best_epoch)
print("Training time:", f"{training_time / 60:.2f} minutes")
print("Best checkpoint:", alexnet_best_path)

Epoch 01/20 | Train Loss: 0.8028 | Val Loss: 0.8954 | Val Acc: 0.2552 | Val F1: 0.1844 | Val AUC: 0.4667 | Time: 54.2s
  Best validation F1 improved. Checkpoint saved.
Epoch 02/20 | Train Loss: 0.8097 | Val Loss: 0.8954 | Val Acc: 0.2552 | Val F1: 0.1844 | Val AUC: 0.4667 | Time: 55.4s
  No improvement for 1/5 epoch(s).
Epoch 03/20 | Train Loss: 0.7970 | Val Loss: 0.8954 | Val Acc: 0.2552 | Val F1: 0.1844 | Val AUC: 0.4667 | Time: 54.5s
  No improvement for 2/5 epoch(s).
Epoch 04/20 | Train Loss: 0.8000 | Val Loss: 0.8954 | Val Acc: 0.2552 | Val F1: 0.1844 | Val AUC: 0.4667 | Time: 54.8s
  No improvement for 3/5 epoch(s).
Epoch 05/20 | Train Loss: 0.8056 | Val Loss: 0.8954 | Val Acc: 0.2552 | Val F1: 0.1844 | Val AUC: 0.4667 | Time: 54.2s
  No improvement for 4/5 epoch(s).
Epoch 06/20 | Train Loss: 0.7960 | Val Loss: 0.8954 | Val Acc: 0.2552 | Val F1: 0.1844 | Val AUC: 0.4667 | Time: 53.7s
  No improvement for 5/5 epoch(s).
Early stopping triggered.
------------------------------------

In [47]:
# --- Diagnose the saved AlexNet checkpoint on the validation split ---

alexnet_checkpoint = torch.load(
    alexnet_best_path,
    map_location=DEVICE,
    weights_only=False
)

alexnet.load_state_dict(
    alexnet_checkpoint["model_state_dict"]
)

alexnet_val_metrics = evaluate_model(
    alexnet,
    val_loader,
    alexnet_criterion,
    DEVICE
)

print("AlexNet checkpoint diagnostic")
print("-" * 50)
print("Checkpoint epoch:", alexnet_checkpoint["epoch"])
print("Checkpoint validation F1:", alexnet_checkpoint["best_val_f1"])
print("Validation loss:", f"{alexnet_val_metrics['loss']:.4f}")
print("Validation accuracy:", f"{alexnet_val_metrics['accuracy']:.4f}")
print("Validation precision:", f"{alexnet_val_metrics['precision']:.4f}")
print("Validation recall:", f"{alexnet_val_metrics['recall']:.4f}")
print("Validation F1:", f"{alexnet_val_metrics['f1']:.4f}")
print("Validation ROC-AUC:", f"{alexnet_val_metrics['roc_auc']:.4f}")
print("Validation confusion matrix:")
print(alexnet_val_metrics["confusion_matrix"])

print("-" * 50)
print("Stored history entries:", len(alexnet_checkpoint["history"]))

AlexNet checkpoint diagnostic
--------------------------------------------------
Checkpoint epoch: 1
Checkpoint validation F1: 0.18441744102930666
Validation loss: 0.8954
Validation accuracy: 0.2552
Validation precision: 0.1052
Validation recall: 0.7457
Validation F1: 0.1844
Validation ROC-AUC: 0.4667
Validation confusion matrix:
[[ 262 1097]
 [  44  129]]
--------------------------------------------------
Stored history entries: 1


In [48]:
# --- Diagnose AlexNet prediction behavior on training and validation data ---

def prediction_diagnostics(model, loader, device):
    model.eval()

    all_labels = []
    all_predictions = []
    all_probabilities = []

    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device, non_blocking=True)

            outputs = model(images)
            probabilities = torch.softmax(outputs, dim=1)[:, 1]
            predictions = torch.argmax(outputs, dim=1)

            all_labels.extend(labels.numpy())
            all_predictions.extend(predictions.cpu().numpy())
            all_probabilities.extend(probabilities.cpu().numpy())

    return (
        np.array(all_labels),
        np.array(all_predictions),
        np.array(all_probabilities)
    )


train_labels_diag, train_predictions_diag, train_probabilities_diag = (
    prediction_diagnostics(
        alexnet,
        train_loader,
        DEVICE
    )
)

val_labels_diag, val_predictions_diag, val_probabilities_diag = (
    prediction_diagnostics(
        alexnet,
        val_loader,
        DEVICE
    )
)

print("AlexNet prediction diagnostics")
print("-" * 60)

print("TRAINING SET")
print("Actual class counts:")
print(np.bincount(train_labels_diag))
print("Predicted class counts:")
print(np.bincount(train_predictions_diag))
print(
    "Probability of melanoma - min:",
    f"{train_probabilities_diag.min():.4f}",
    "| max:",
    f"{train_probabilities_diag.max():.4f}",
    "| mean:",
    f"{train_probabilities_diag.mean():.4f}"
)

print("-" * 60)

print("VALIDATION SET")
print("Actual class counts:")
print(np.bincount(val_labels_diag))
print("Predicted class counts:")
print(np.bincount(val_predictions_diag))
print(
    "Probability of melanoma - min:",
    f"{val_probabilities_diag.min():.4f}",
    "| max:",
    f"{val_probabilities_diag.max():.4f}",
    "| mean:",
    f"{val_probabilities_diag.mean():.4f}"
)

print("-" * 60)

print("Training prediction confusion matrix:")
print(confusion_matrix(
    train_labels_diag,
    train_predictions_diag
))

print("-" * 60)

print("Validation prediction confusion matrix:")
print(confusion_matrix(
    val_labels_diag,
    val_predictions_diag
))

AlexNet prediction diagnostics
------------------------------------------------------------
TRAINING SET
Actual class counts:
[6208  773]
Predicted class counts:
[1444 5537]
Probability of melanoma - min: 0.1703 | max: 0.9723 | mean: 0.6177
------------------------------------------------------------
VALIDATION SET
Actual class counts:
[1359  173]
Predicted class counts:
[ 306 1226]
Probability of melanoma - min: 0.3103 | max: 0.9372 | mean: 0.5925
------------------------------------------------------------
Training prediction confusion matrix:
[[1275 4933]
 [ 169  604]]
------------------------------------------------------------
Validation prediction confusion matrix:
[[ 262 1097]
 [  44  129]]


In [49]:
# --- Inspect AlexNet trainable parameters and current output behavior ---

print("AlexNet model state inspection")
print("-" * 60)

trainable_parameters = sum(
    parameter.numel()
    for parameter in alexnet.parameters()
    if parameter.requires_grad
)

total_parameters = sum(
    parameter.numel()
    for parameter in alexnet.parameters()
)

print("Total parameters:", f"{total_parameters:,}")
print("Trainable parameters:", f"{trainable_parameters:,}")

if isinstance(alexnet, nn.DataParallel):
    alexnet_core = alexnet.module
else:
    alexnet_core = alexnet

print("-" * 60)
print("Classifier:")
print(alexnet_core.classifier)

print("-" * 60)

images_sample, labels_sample = next(iter(val_loader))

images_sample = images_sample.to(
    DEVICE,
    non_blocking=True
)

with torch.no_grad():
    sample_outputs = alexnet(images_sample)
    sample_probabilities = torch.softmax(
        sample_outputs,
        dim=1
    )

print("Sample output inspection")
print("Logit minimum:", f"{sample_outputs.min().item():.4f}")
print("Logit maximum:", f"{sample_outputs.max().item():.4f}")
print("Melanoma probability minimum:", f"{sample_probabilities[:, 1].min().item():.4f}")
print("Melanoma probability maximum:", f"{sample_probabilities[:, 1].max().item():.4f}")
print("Melanoma probability mean:", f"{sample_probabilities[:, 1].mean().item():.4f}")

print("-" * 60)
print("All AlexNet parameters are trainable:",
      trainable_parameters == total_parameters)

AlexNet model state inspection
------------------------------------------------------------
Total parameters: 57,012,034
Trainable parameters: 57,012,034
------------------------------------------------------------
Classifier:
Sequential(
  (0): Dropout(p=0.5, inplace=False)
  (1): Linear(in_features=9216, out_features=4096, bias=True)
  (2): ReLU(inplace=True)
  (3): Dropout(p=0.5, inplace=False)
  (4): Linear(in_features=4096, out_features=4096, bias=True)
  (5): ReLU(inplace=True)
  (6): Linear(in_features=4096, out_features=2, bias=True)
)
------------------------------------------------------------
Sample output inspection
Logit minimum: -0.4112
Logit maximum: 2.1030
Melanoma probability minimum: 0.3998
Melanoma probability maximum: 0.8627
Melanoma probability mean: 0.5906
------------------------------------------------------------
All AlexNet parameters are trainable: True


In [50]:
# --- Evaluate a fresh pretrained AlexNet before fine-tuning ---

fresh_alexnet = models.alexnet(
    weights=models.AlexNet_Weights.DEFAULT
)

fresh_alexnet.classifier[6] = nn.Linear(
    fresh_alexnet.classifier[6].in_features,
    NUM_CLASSES
)

fresh_alexnet = fresh_alexnet.to(DEVICE)

if torch.cuda.device_count() > 1:
    fresh_alexnet = nn.DataParallel(fresh_alexnet)

fresh_alexnet_metrics = evaluate_model(
    fresh_alexnet,
    val_loader,
    alexnet_criterion,
    DEVICE
)

print("Fresh pretrained AlexNet validation diagnostic")
print("-" * 60)
print("Validation loss:", f"{fresh_alexnet_metrics['loss']:.4f}")
print("Validation accuracy:", f"{fresh_alexnet_metrics['accuracy']:.4f}")
print("Validation precision:", f"{fresh_alexnet_metrics['precision']:.4f}")
print("Validation recall:", f"{fresh_alexnet_metrics['recall']:.4f}")
print("Validation F1:", f"{fresh_alexnet_metrics['f1']:.4f}")
print("Validation ROC-AUC:", f"{fresh_alexnet_metrics['roc_auc']:.4f}")
print("Validation confusion matrix:")
print(fresh_alexnet_metrics["confusion_matrix"])

Fresh pretrained AlexNet validation diagnostic
------------------------------------------------------------
Validation loss: 0.6582
Validation accuracy: 0.6155
Validation precision: 0.1176
Validation recall: 0.3699
Validation F1: 0.1785
Validation ROC-AUC: 0.5011
Validation confusion matrix:
[[879 480]
 [109  64]]


In [51]:
# --- Verify HAM10000 classification inputs before further training ---

images_check, labels_check = next(iter(train_loader))

print("HAM10000 model input verification")
print("-" * 60)
print("Batch shape:", tuple(images_check.shape))
print("Data type:", images_check.dtype)
print("Input minimum:", f"{images_check.min().item():.4f}")
print("Input maximum:", f"{images_check.max().item():.4f}")
print("Input mean:", f"{images_check.mean().item():.4f}")
print("Input standard deviation:", f"{images_check.std().item():.4f}")
print("Label shape:", tuple(labels_check.shape))
print("Labels present:", torch.unique(labels_check).tolist())

print("-" * 60)
print("Expected input format")
print("Shape: [batch, 3, 224, 224]")
print("ImageNet normalization: applied")
print("Original array range before normalization: [0, 1]")
print("Additional division by 255: not applied")

HAM10000 model input verification
------------------------------------------------------------
Batch shape: (32, 3, 224, 224)
Data type: torch.float32
Input minimum: -2.1179
Input maximum: 2.6162
Input mean: 0.5166
Input standard deviation: 0.9622
Label shape: (32,)
Labels present: [0, 1]
------------------------------------------------------------
Expected input format
Shape: [batch, 3, 224, 224]
ImageNet normalization: applied
Original array range before normalization: [0, 1]
Additional division by 255: not applied


In [52]:
# --- Create a balanced training sampler for the controlled AlexNet retry ---

from torch.utils.data import WeightedRandomSampler

train_class_counts = np.bincount(
    train_dataset.labels,
    minlength=NUM_CLASSES
)

sample_weights = np.array([
    1.0 / train_class_counts[label]
    for label in train_dataset.labels
])

sample_weights = torch.as_tensor(
    sample_weights,
    dtype=torch.double
)

alexnet_sampler = WeightedRandomSampler(
    weights=sample_weights,
    num_samples=len(sample_weights),
    replacement=True,
    generator=torch.Generator().manual_seed(SEED)
)

alexnet_balanced_train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    sampler=alexnet_sampler,
    num_workers=NUM_WORKERS,
    pin_memory=True
)

print("Balanced AlexNet training loader created.")
print("-" * 60)
print("Original training counts:")
print(
    "Non-melanoma:",
    train_class_counts[0]
)
print(
    "Melanoma:",
    train_class_counts[1]
)
print("Sampling method:", "WeightedRandomSampler")
print("Replacement:", True)
print("Samples per epoch:", len(alexnet_sampler))

Balanced AlexNet training loader created.
------------------------------------------------------------
Original training counts:
Non-melanoma: 6208
Melanoma: 773
Sampling method: WeightedRandomSampler
Replacement: True
Samples per epoch: 6981


In [53]:
# --- Create a fresh AlexNet for the controlled balanced-sampling experiment ---

alexnet_retry = models.alexnet(
    weights=models.AlexNet_Weights.DEFAULT
)

alexnet_retry.classifier[6] = nn.Linear(
    alexnet_retry.classifier[6].in_features,
    NUM_CLASSES
)

alexnet_retry = alexnet_retry.to(DEVICE)

if torch.cuda.device_count() > 1:
    alexnet_retry = nn.DataParallel(alexnet_retry)

alexnet_retry_parameter_count = sum(
    parameter.numel()
    for parameter in alexnet_retry.parameters()
)

print("Fresh AlexNet retry model created.")
print("-" * 60)
print("Parameters:", f"{alexnet_retry_parameter_count:,}")
print("Device:", DEVICE)
print(
    "Multi-GPU:",
    "Enabled" if isinstance(alexnet_retry, nn.DataParallel) else "Disabled"
)
print("Initialization:", "ImageNet pretrained")

Fresh AlexNet retry model created.
------------------------------------------------------------
Parameters: 57,012,034
Device: cuda
Multi-GPU: Enabled
Initialization: ImageNet pretrained


In [54]:
# --- Configure loss and optimizer for the balanced-sampling AlexNet retry ---

alexnet_retry_criterion = nn.CrossEntropyLoss()

alexnet_retry_optimizer = torch.optim.Adam(
    alexnet_retry.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY
)

print("AlexNet retry training configuration created.")
print("-" * 60)
print("Loss:", "CrossEntropyLoss")
print("Class weighting:", "Disabled")
print("Sampling:", "WeightedRandomSampler")
print("Optimizer:", "Adam")
print("Learning rate:", LEARNING_RATE)
print("Weight decay:", WEIGHT_DECAY)

AlexNet retry training configuration created.
------------------------------------------------------------
Loss: CrossEntropyLoss
Class weighting: Disabled
Sampling: WeightedRandomSampler
Optimizer: Adam
Learning rate: 0.0001
Weight decay: 1e-05


In [56]:
# --- Define training constants required for the AlexNet retry ---

MAX_EPOCHS = 20
EARLY_STOPPING_PATIENCE = 5

print("AlexNet retry training constants restored.")
print("-" * 60)
print("Maximum epochs:", MAX_EPOCHS)
print("Early stopping patience:", EARLY_STOPPING_PATIENCE)

AlexNet retry training constants restored.
------------------------------------------------------------
Maximum epochs: 20
Early stopping patience: 5


In [57]:
# --- Train the balanced-sampling AlexNet retry ---

alexnet_retry_history = []
alexnet_retry_best_f1 = -1.0
alexnet_retry_best_epoch = 0
alexnet_retry_patience_counter = 0

alexnet_retry_checkpoint_path = "/kaggle/working/ham10000_alexnet_balanced_best.pth"

alexnet_retry_start_time = time.time()

for epoch in range(1, MAX_EPOCHS + 1):
    epoch_start = time.time()

    alexnet_retry.train()
    train_loss_sum = 0.0
    train_samples = 0

    for images, labels in alexnet_balanced_train_loader:
        images = images.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)

        alexnet_retry_optimizer.zero_grad(set_to_none=True)

        outputs = alexnet_retry(images)
        loss = alexnet_retry_criterion(outputs, labels)

        loss.backward()
        alexnet_retry_optimizer.step()

        batch_size = labels.size(0)
        train_loss_sum += loss.item() * batch_size
        train_samples += batch_size

    train_loss = train_loss_sum / train_samples

    val_metrics = evaluate_model(
        alexnet_retry,
        val_loader,
        alexnet_retry_criterion,
        DEVICE
    )

    epoch_time = time.time() - epoch_start

    epoch_record = {
        "epoch": epoch,
        "train_loss": train_loss,
        "val_loss": val_metrics["loss"],
        "val_accuracy": val_metrics["accuracy"],
        "val_precision": val_metrics["precision"],
        "val_recall": val_metrics["recall"],
        "val_f1": val_metrics["f1"],
        "val_roc_auc": val_metrics["roc_auc"],
        "epoch_time_seconds": epoch_time
    }

    alexnet_retry_history.append(epoch_record)

    print(
        f"Epoch {epoch:02d}/{MAX_EPOCHS} | "
        f"Train Loss {train_loss:.4f} | "
        f"Val Loss {val_metrics['loss']:.4f} | "
        f"Val Acc {val_metrics['accuracy']:.4f} | "
        f"Val F1 {val_metrics['f1']:.4f} | "
        f"Val AUC {val_metrics['roc_auc']:.4f} | "
        f"Time {epoch_time:.1f}s"
    )

    if val_metrics["f1"] > alexnet_retry_best_f1:
        alexnet_retry_best_f1 = val_metrics["f1"]
        alexnet_retry_best_epoch = epoch
        alexnet_retry_patience_counter = 0

        torch.save(
            {
                "epoch": epoch,
                "model_state_dict": alexnet_retry.state_dict(),
                "optimizer_state_dict": alexnet_retry_optimizer.state_dict(),
                "best_val_f1": alexnet_retry_best_f1,
                "val_metrics": val_metrics,
                "history": alexnet_retry_history
            },
            alexnet_retry_checkpoint_path
        )

        print("  Best checkpoint saved.")
    else:
        alexnet_retry_patience_counter += 1

        if alexnet_retry_patience_counter >= EARLY_STOPPING_PATIENCE:
            print(
                f"  Early stopping triggered after "
                f"{EARLY_STOPPING_PATIENCE} epochs without improvement."
            )
            break

alexnet_retry_total_time = time.time() - alexnet_retry_start_time

print("\n" + "=" * 60)
print("Balanced AlexNet training completed.")
print("Best epoch:", alexnet_retry_best_epoch)
print(f"Best validation F1: {alexnet_retry_best_f1:.4f}")
print(f"Total training time: {alexnet_retry_total_time / 60:.2f} min")
print("Checkpoint:", alexnet_retry_checkpoint_path)

Epoch 01/20 | Train Loss 0.4650 | Val Loss 0.5514 | Val Acc 0.7245 | Val F1 0.4282 | Val AUC 0.9010 | Time 55.9s
  Best checkpoint saved.
Epoch 02/20 | Train Loss 0.3834 | Val Loss 0.5284 | Val Acc 0.7180 | Val F1 0.4240 | Val AUC 0.9060 | Time 55.3s
Epoch 03/20 | Train Loss 0.3673 | Val Loss 0.3438 | Val Acc 0.8283 | Val F1 0.5244 | Val AUC 0.9171 | Time 55.0s
  Best checkpoint saved.
Epoch 04/20 | Train Loss 0.3416 | Val Loss 0.4389 | Val Acc 0.7761 | Val F1 0.4747 | Val AUC 0.8966 | Time 55.3s
Epoch 05/20 | Train Loss 0.2983 | Val Loss 0.3785 | Val Acc 0.8166 | Val F1 0.5180 | Val AUC 0.9007 | Time 54.9s
Epoch 06/20 | Train Loss 0.2858 | Val Loss 0.3421 | Val Acc 0.8525 | Val F1 0.5129 | Val AUC 0.8885 | Time 55.2s
Epoch 07/20 | Train Loss 0.2584 | Val Loss 0.4354 | Val Acc 0.8244 | Val F1 0.5272 | Val AUC 0.9077 | Time 54.7s
  Best checkpoint saved.
Epoch 08/20 | Train Loss 0.2365 | Val Loss 0.3182 | Val Acc 0.8590 | Val F1 0.5481 | Val AUC 0.9090 | Time 54.6s
  Best checkpoint sav

In [59]:
# --- Reload the trusted best AlexNet checkpoint with full checkpoint data ---

alexnet_retry_checkpoint = torch.load(
    alexnet_retry_checkpoint_path,
    map_location=DEVICE,
    weights_only=False
)

alexnet_retry.load_state_dict(
    alexnet_retry_checkpoint["model_state_dict"]
)

alexnet_retry_best_epoch = alexnet_retry_checkpoint["epoch"]
alexnet_retry_best_val_f1 = alexnet_retry_checkpoint["best_val_f1"]

print("Best AlexNet checkpoint reloaded successfully.")
print("-" * 60)
print("Best epoch:", alexnet_retry_best_epoch)
print(f"Best validation F1: {alexnet_retry_best_val_f1:.4f}")
print("Checkpoint path:", alexnet_retry_checkpoint_path)

Best AlexNet checkpoint reloaded successfully.
------------------------------------------------------------
Best epoch: 8
Best validation F1: 0.5481
Checkpoint path: /kaggle/working/ham10000_alexnet_balanced_best.pth


In [60]:
# --- Evaluate the selected balanced AlexNet on the untouched HAM10000 test set ---

alexnet_retry_test_metrics = evaluate_model(
    alexnet_retry,
    test_loader,
    alexnet_retry_criterion,
    DEVICE
)

print("Balanced AlexNet test evaluation completed.")
print("-" * 60)
print(f"Test Loss:     {alexnet_retry_test_metrics['loss']:.6f}")
print(f"Accuracy:      {alexnet_retry_test_metrics['accuracy']:.6f}")
print(f"Precision:     {alexnet_retry_test_metrics['precision']:.6f}")
print(f"Recall:        {alexnet_retry_test_metrics['recall']:.6f}")
print(f"F1 Score:      {alexnet_retry_test_metrics['f1']:.6f}")
print(f"ROC-AUC:       {alexnet_retry_test_metrics['roc_auc']:.6f}")
print("Confusion Matrix:")
print(alexnet_retry_test_metrics["confusion_matrix"])

Balanced AlexNet test evaluation completed.
------------------------------------------------------------
Test Loss:     0.386433
Accuracy:      0.845539
Precision:     0.385965
Recall:        0.658683
F1 Score:      0.486726
ROC-AUC:       0.870407
Confusion Matrix:
[[1160  175]
 [  57  110]]


In [62]:
# --- Save the complete balanced AlexNet experiment record ---

import json

alexnet_retry_results = {
    "model": "AlexNet",
    "dataset": "HAM10000",
    "task": "Binary melanoma classification",
    "input_size": "224x224",
    "initialization": "ImageNet pretrained",
    "sampling": "WeightedRandomSampler",
    "loss": "CrossEntropyLoss",
    "class_weighting": False,
    "optimizer": "Adam",
    "learning_rate": LEARNING_RATE,
    "weight_decay": WEIGHT_DECAY,
    "max_epochs": MAX_EPOCHS,
    "early_stopping_patience": EARLY_STOPPING_PATIENCE,
    "best_epoch": alexnet_retry_best_epoch,
    "best_validation_f1": alexnet_retry_best_val_f1,
    "test_metrics": alexnet_retry_test_metrics,
    "training_time_minutes": alexnet_retry_total_time / 60,
    "checkpoint": alexnet_retry_checkpoint_path
}

alexnet_retry_results_path = (
    "/kaggle/working/ham10000_alexnet_balanced_results.json"
)

with open(alexnet_retry_results_path, "w") as f:
    json.dump(
        alexnet_retry_results,
        f,
        indent=2,
        default=lambda value: value.tolist()
        if isinstance(value, np.ndarray)
        else value
    )

alexnet_retry_history_path = (
    "/kaggle/working/ham10000_alexnet_balanced_training_history.pt"
)

torch.save(
    alexnet_retry_history,
    alexnet_retry_history_path
)

print("AlexNet experiment record saved.")
print("-" * 60)
print("Results:", alexnet_retry_results_path)
print("History:", alexnet_retry_history_path)
print("Checkpoint:", alexnet_retry_checkpoint_path)

AlexNet experiment record saved.
------------------------------------------------------------
Results: /kaggle/working/ham10000_alexnet_balanced_results.json
History: /kaggle/working/ham10000_alexnet_balanced_training_history.pt
Checkpoint: /kaggle/working/ham10000_alexnet_balanced_best.pth


In [63]:
# --- Create a backup archive for the completed AlexNet experiment ---

import zipfile
from pathlib import Path

alexnet_backup_path = Path(
    "/kaggle/working/HAM10000_AlexNet_Balanced_Baseline_Backup.zip"
)

alexnet_backup_files = [
    Path(alexnet_retry_checkpoint_path),
    Path(alexnet_retry_history_path),
    Path(alexnet_retry_results_path)
]

with zipfile.ZipFile(
    alexnet_backup_path,
    mode="w",
    compression=zipfile.ZIP_DEFLATED
) as zip_file:

    for file_path in alexnet_backup_files:
        zip_file.write(
            file_path,
            arcname=file_path.name
        )

print("AlexNet backup created successfully.")
print("-" * 60)
print("Backup:", alexnet_backup_path)
print(
    f"Backup size: "
    f"{alexnet_backup_path.stat().st_size / (1024 ** 2):.2f} MB"
)
print("Files included:")
for file_path in alexnet_backup_files:
    print(" -", file_path.name)

AlexNet backup created successfully.
------------------------------------------------------------
Backup: /kaggle/working/HAM10000_AlexNet_Balanced_Baseline_Backup.zip
Backup size: 611.28 MB
Files included:
 - ham10000_alexnet_balanced_best.pth
 - ham10000_alexnet_balanced_training_history.pt
 - ham10000_alexnet_balanced_results.json


In [64]:
# --- Verify all completed AlexNet artifacts before creating smaller downloads ---

from pathlib import Path

alexnet_files_to_check = [
    Path("/kaggle/working/ham10000_alexnet_balanced_best.pth"),
    Path("/kaggle/working/ham10000_alexnet_balanced_training_history.pt"),
    Path("/kaggle/working/ham10000_alexnet_balanced_results.json"),
    Path("/kaggle/working/HAM10000_AlexNet_Balanced_Baseline_Backup.zip")
]

print("AlexNet artifact verification")
print("-" * 60)

for file_path in alexnet_files_to_check:
    if file_path.exists():
        size_mb = file_path.stat().st_size / (1024 ** 2)
        print(f"FOUND  | {file_path.name} | {size_mb:.2f} MB")
    else:
        print(f"MISSING | {file_path.name}")

AlexNet artifact verification
------------------------------------------------------------
FOUND  | ham10000_alexnet_balanced_best.pth | 652.47 MB
FOUND  | ham10000_alexnet_balanced_training_history.pt | 0.00 MB
FOUND  | ham10000_alexnet_balanced_results.json | 0.00 MB
FOUND  | HAM10000_AlexNet_Balanced_Baseline_Backup.zip | 611.28 MB


In [65]:
# --- Split the AlexNet backup into smaller downloadable parts ---

from pathlib import Path

backup_source = Path(
    "/kaggle/working/HAM10000_AlexNet_Balanced_Baseline_Backup.zip"
)

part_size = 100 * 1024 * 1024

with open(backup_source, "rb") as source_file:
    part_number = 1

    while True:
        data = source_file.read(part_size)

        if not data:
            break

        part_path = Path(
            f"/kaggle/working/"
            f"HAM10000_AlexNet_Balanced_Baseline_Backup.part{part_number:02d}"
        )

        with open(part_path, "wb") as part_file:
            part_file.write(data)

        print(
            f"Created {part_path.name} | "
            f"{len(data) / (1024 ** 2):.2f} MB"
        )

        part_number += 1

print("-" * 60)
print(f"Total parts created: {part_number - 1}")

Created HAM10000_AlexNet_Balanced_Baseline_Backup.part01 | 100.00 MB
Created HAM10000_AlexNet_Balanced_Baseline_Backup.part02 | 100.00 MB
Created HAM10000_AlexNet_Balanced_Baseline_Backup.part03 | 100.00 MB
Created HAM10000_AlexNet_Balanced_Baseline_Backup.part04 | 100.00 MB
Created HAM10000_AlexNet_Balanced_Baseline_Backup.part05 | 100.00 MB
Created HAM10000_AlexNet_Balanced_Baseline_Backup.part06 | 100.00 MB
Created HAM10000_AlexNet_Balanced_Baseline_Backup.part07 | 11.28 MB
------------------------------------------------------------
Total parts created: 7


In [57]:
# --- Create the ImageNet-pretrained VGG16 benchmark model ---

from torchvision import models
import torch.nn as nn

vgg16_model = models.vgg16(weights=models.VGG16_Weights.DEFAULT)

vgg16_model.classifier[6] = nn.Linear(
    vgg16_model.classifier[6].in_features,
    NUM_CLASSES
)

vgg16_model = vgg16_model.to(DEVICE)

if torch.cuda.device_count() > 1:
    vgg16_model = nn.DataParallel(vgg16_model)

print("VGG16 model created successfully.")
print("-" * 60)
print(f"Parameters: {sum(p.numel() for p in vgg16_model.parameters()):,}")
print(f"Trainable parameters: {sum(p.numel() for p in vgg16_model.parameters() if p.requires_grad):,}")
print(f"Device: {DEVICE}")
print(f"Multi-GPU: {'Enabled' if torch.cuda.device_count() > 1 else 'Disabled'}")
print("Initialization: ImageNet pretrained")

VGG16 model created successfully.
------------------------------------------------------------
Parameters: 134,268,738
Trainable parameters: 134,268,738
Device: cuda
Multi-GPU: Enabled
Initialization: ImageNet pretrained


In [58]:
# --- Configure the VGG16 loss function and optimizer ---

vgg16_criterion = nn.CrossEntropyLoss()

vgg16_optimizer = torch.optim.Adam(
    vgg16_model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY
)

print("VGG16 training configuration created.")
print("-" * 60)
print("Loss: CrossEntropyLoss")
print("Class weighting: Disabled")
print("Sampling: WeightedRandomSampler")
print(f"Optimizer: Adam")
print(f"Learning rate: {LEARNING_RATE}")
print(f"Weight decay: {WEIGHT_DECAY}")

VGG16 training configuration created.
------------------------------------------------------------
Loss: CrossEntropyLoss
Class weighting: Disabled
Sampling: WeightedRandomSampler
Optimizer: Adam
Learning rate: 0.0001
Weight decay: 1e-05


In [61]:
# Recreate the balanced training loader for VGG16

from torch.utils.data import WeightedRandomSampler, DataLoader
import numpy as np

train_labels = np.array(train_dataset.labels)

class_counts = np.bincount(train_labels)
class_weights = 1.0 / class_counts

sample_weights = class_weights[train_labels]
sample_weights = torch.as_tensor(sample_weights, dtype=torch.double)

balanced_sampler = WeightedRandomSampler(
    weights=sample_weights,
    num_samples=len(train_labels),
    replacement=True
)

balanced_train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    sampler=balanced_sampler,
    num_workers=NUM_WORKERS,
    pin_memory=True
)

print("Balanced training loader recreated successfully.")
print("Samples per epoch:", len(balanced_train_loader.dataset))
print("Batches per epoch:", len(balanced_train_loader))
print("Original class counts:", class_counts)
print("Sampling weights:", class_weights)

Balanced training loader recreated successfully.
Samples per epoch: 6981
Batches per epoch: 219
Original class counts: [6208  773]
Sampling weights: [0.00016108 0.00129366]


In [62]:
# --- Train VGG16 with balanced sampling and validation-based checkpointing ---

import time
import copy

vgg16_best_f1 = -1.0
vgg16_best_epoch = 0
vgg16_best_state = None
vgg16_history = []
vgg16_patience_counter = 0

VGG16_CHECKPOINT_PATH = "/kaggle/working/ham10000_vgg16_balanced_best.pth"

training_start = time.time()

for epoch in range(1, MAX_EPOCHS + 1):

    epoch_start = time.time()

    vgg16_model.train()

    running_loss = 0.0
    training_samples = 0

    for images, labels in balanced_train_loader:
        images = images.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)

        vgg16_optimizer.zero_grad()

        outputs = vgg16_model(images)
        loss = vgg16_criterion(outputs, labels)

        loss.backward()
        vgg16_optimizer.step()

        batch_size = images.size(0)
        running_loss += loss.item() * batch_size
        training_samples += batch_size

    train_loss = running_loss / training_samples

    val_metrics = evaluate_model(
        vgg16_model,
        val_loader,
        vgg16_criterion,
        DEVICE
    )

    epoch_time = time.time() - epoch_start

    vgg16_history.append({
        "epoch": epoch,
        "train_loss": train_loss,
        **val_metrics,
        "epoch_time_seconds": epoch_time
    })

    current_f1 = val_metrics["f1"]

    print(
        f"Epoch {epoch:02d}/{MAX_EPOCHS} | "
        f"Train Loss {train_loss:.4f} | "
        f"Val Loss {val_metrics['loss']:.4f} | "
        f"Val Acc {val_metrics['accuracy']:.4f} | "
        f"Val F1 {val_metrics['f1']:.4f} | "
        f"Val AUC {val_metrics['roc_auc']:.4f} | "
        f"Time {epoch_time:.1f}s"
    )

    if current_f1 > vgg16_best_f1:
        vgg16_best_f1 = current_f1
        vgg16_best_epoch = epoch
        vgg16_best_state = copy.deepcopy(vgg16_model.state_dict())
        vgg16_patience_counter = 0

        torch.save(
            {
                "epoch": epoch,
                "model_state_dict": vgg16_best_state,
                "optimizer_state_dict": vgg16_optimizer.state_dict(),
                "best_val_f1": vgg16_best_f1,
                "history": vgg16_history,
            },
            VGG16_CHECKPOINT_PATH
        )

        print(f"  Best checkpoint saved at epoch {epoch}.")

    else:
        vgg16_patience_counter += 1

        if vgg16_patience_counter >= EARLY_STOPPING_PATIENCE:
            print(
                f"Early stopping triggered after "
                f"{EARLY_STOPPING_PATIENCE} epochs without improvement."
            )
            break

total_training_time = time.time() - training_start

print("-" * 60)
print("VGG16 training completed.")
print(f"Best epoch: {vgg16_best_epoch}")
print(f"Best validation F1: {vgg16_best_f1:.4f}")
print(f"Total training time: {total_training_time / 60:.2f} minutes")
print(f"Checkpoint: {VGG16_CHECKPOINT_PATH}")

Epoch 01/20 | Train Loss 0.4908 | Val Loss 0.4358 | Val Acc 0.7905 | Val F1 0.4763 | Val AUC 0.8944 | Time 91.1s
  Best checkpoint saved at epoch 1.
Epoch 02/20 | Train Loss 0.4313 | Val Loss 0.4457 | Val Acc 0.7082 | Val F1 0.4262 | Val AUC 0.9277 | Time 90.7s
Epoch 03/20 | Train Loss 0.3700 | Val Loss 0.4183 | Val Acc 0.7454 | Val F1 0.4476 | Val AUC 0.9198 | Time 90.4s
Epoch 04/20 | Train Loss 0.3283 | Val Loss 0.4258 | Val Acc 0.7657 | Val F1 0.4697 | Val AUC 0.9146 | Time 90.3s
Epoch 05/20 | Train Loss 0.3262 | Val Loss 0.4878 | Val Acc 0.7748 | Val F1 0.4843 | Val AUC 0.9300 | Time 90.3s
  Best checkpoint saved at epoch 5.
Epoch 06/20 | Train Loss 0.2868 | Val Loss 0.5719 | Val Acc 0.7167 | Val F1 0.4274 | Val AUC 0.9163 | Time 90.7s
Epoch 07/20 | Train Loss 0.2474 | Val Loss 0.6068 | Val Acc 0.7350 | Val F1 0.4469 | Val AUC 0.9253 | Time 90.3s
Epoch 08/20 | Train Loss 0.2341 | Val Loss 0.4088 | Val Acc 0.7957 | Val F1 0.4943 | Val AUC 0.9156 | Time 90.2s
  Best checkpoint saved 

In [63]:
# Load the best VGG16 checkpoint selected by validation F1

checkpoint = torch.load(
    "/kaggle/working/ham10000_vgg16_balanced_best.pth",
    map_location=DEVICE,
    weights_only=False
)

vgg16_model.load_state_dict(checkpoint["model_state_dict"])

vgg16_best_epoch = checkpoint["epoch"]
vgg16_best_f1 = checkpoint["best_val_f1"]

print("Best VGG16 checkpoint loaded successfully.")
print("Best epoch:", vgg16_best_epoch)
print(f"Best validation F1: {vgg16_best_f1:.4f}")

Best VGG16 checkpoint loaded successfully.
Best epoch: 10
Best validation F1: 0.6009


In [64]:
# Evaluate the best VGG16 checkpoint on the untouched HAM10000 test set

vgg16_test_results = evaluate_model(
    vgg16_model,
    test_loader,
    vgg16_criterion,
    DEVICE
)

print("\nVGG16 Official Test Results")
print("-" * 40)
print(f"Loss:       {vgg16_test_results['loss']:.6f}")
print(f"Accuracy:   {vgg16_test_results['accuracy']:.6f}")
print(f"Precision:  {vgg16_test_results['precision']:.6f}")
print(f"Recall:     {vgg16_test_results['recall']:.6f}")
print(f"F1 Score:   {vgg16_test_results['f1']:.6f}")
print(f"ROC-AUC:    {vgg16_test_results['roc_auc']:.6f}")

print("\nConfusion Matrix")
print(vgg16_test_results["confusion_matrix"])


VGG16 Official Test Results
----------------------------------------
Loss:       0.375246
Accuracy:   0.856192
Precision:  0.415225
Recall:     0.718563
F1 Score:   0.526316
ROC-AUC:    0.896064

Confusion Matrix
[[1166  169]
 [  47  120]]


In [66]:
# Save the official VGG16 results in JSON-compatible format

import json
import torch
import numpy as np

vgg16_results_serializable = {}

for key, value in vgg16_test_results.items():
    if isinstance(value, np.ndarray):
        vgg16_results_serializable[key] = value.tolist()
    elif isinstance(value, (np.integer, np.floating)):
        vgg16_results_serializable[key] = value.item()
    else:
        vgg16_results_serializable[key] = value

vgg16_results_path = "/kaggle/working/ham10000_vgg16_balanced_results.json"
vgg16_history_path = "/kaggle/working/ham10000_vgg16_balanced_training_history.pt"

with open(vgg16_results_path, "w") as f:
    json.dump(vgg16_results_serializable, f, indent=4)

torch.save(
    {
        "history": vgg16_history,
        "best_epoch": vgg16_best_epoch,
        "best_val_f1": vgg16_best_f1,
        "test_results": vgg16_results_serializable
    },
    vgg16_history_path
)

print("VGG16 results and history saved successfully.")
print(vgg16_results_path)
print(vgg16_history_path)

VGG16 results and history saved successfully.
/kaggle/working/ham10000_vgg16_balanced_results.json
/kaggle/working/ham10000_vgg16_balanced_training_history.pt


In [67]:
# Verify the VGG16 model, results, and training-history artifacts

import os

vgg16_artifacts = [
    "/kaggle/working/ham10000_vgg16_balanced_best.pth",
    "/kaggle/working/ham10000_vgg16_balanced_training_history.pt",
    "/kaggle/working/ham10000_vgg16_balanced_results.json"
]

print("VGG16 Artifact Verification")
print("-" * 50)

for path in vgg16_artifacts:
    if os.path.exists(path):
        size_mb = os.path.getsize(path) / (1024 ** 2)
        print(f"✓ {os.path.basename(path)}")
        print(f"  Size: {size_mb:.2f} MB")
    else:
        print(f"✗ MISSING: {os.path.basename(path)}")

VGG16 Artifact Verification
--------------------------------------------------
✓ ham10000_vgg16_balanced_best.pth
  Size: 1536.63 MB
✓ ham10000_vgg16_balanced_training_history.pt
  Size: 0.01 MB
✓ ham10000_vgg16_balanced_results.json
  Size: 0.00 MB


In [68]:
# Create a complete VGG16 baseline backup package

import os
import zipfile

vgg16_backup_path = "/kaggle/working/HAM10000_VGG16_Balanced_Baseline_Backup.zip"

vgg16_files = [
    "/kaggle/working/ham10000_vgg16_balanced_best.pth",
    "/kaggle/working/ham10000_vgg16_balanced_training_history.pt",
    "/kaggle/working/ham10000_vgg16_balanced_results.json"
]

print("Creating VGG16 backup...")
print()

with zipfile.ZipFile(
    vgg16_backup_path,
    mode="w",
    compression=zipfile.ZIP_STORED
) as backup:
    for file_path in vgg16_files:
        print(f"Adding: {os.path.basename(file_path)}")
        backup.write(
            file_path,
            arcname=os.path.basename(file_path)
        )

backup_size_gb = os.path.getsize(vgg16_backup_path) / (1024 ** 3)

print()
print("VGG16 backup created successfully.")
print(f"Backup path: {vgg16_backup_path}")
print(f"Backup size: {backup_size_gb:.2f} GB")

Creating VGG16 backup...

Adding: ham10000_vgg16_balanced_best.pth
Adding: ham10000_vgg16_balanced_training_history.pt
Adding: ham10000_vgg16_balanced_results.json

VGG16 backup created successfully.
Backup path: /kaggle/working/HAM10000_VGG16_Balanced_Baseline_Backup.zip
Backup size: 1.50 GB


In [69]:
# Verify that the VGG16 backup contains all required artifacts

import os
import zipfile

vgg16_backup_path = "/kaggle/working/HAM10000_VGG16_Balanced_Baseline_Backup.zip"

expected_files = {
    "ham10000_vgg16_balanced_best.pth",
    "ham10000_vgg16_balanced_training_history.pt",
    "ham10000_vgg16_balanced_results.json"
}

with zipfile.ZipFile(vgg16_backup_path, "r") as backup:
    backup_files = set(backup.namelist())

print("VGG16 Backup Verification")
print("-" * 50)

for filename in sorted(expected_files):
    if filename in backup_files:
        info = backup.getinfo(filename)
        size_mb = info.file_size / (1024 ** 2)
        print(f"✓ {filename}")
        print(f"  Size inside ZIP: {size_mb:.2f} MB")
    else:
        print(f"✗ MISSING: {filename}")

print()
print("All required files present:", expected_files.issubset(backup_files))
print(f"ZIP size: {os.path.getsize(vgg16_backup_path) / (1024 ** 3):.2f} GB")

VGG16 Backup Verification
--------------------------------------------------
✓ ham10000_vgg16_balanced_best.pth
  Size inside ZIP: 1536.63 MB
✓ ham10000_vgg16_balanced_results.json
  Size inside ZIP: 0.00 MB
✓ ham10000_vgg16_balanced_training_history.pt
  Size inside ZIP: 0.01 MB

All required files present: True
ZIP size: 1.50 GB


In [69]:
# --- Recreate the balanced training loader for the VGG16 benchmark ---

from torch.utils.data import WeightedRandomSampler, DataLoader

train_labels = train_dataset.labels

class_counts = np.bincount(train_labels)

sample_weights = np.zeros(len(train_labels), dtype=np.float64)

for class_index in range(NUM_CLASSES):
    class_weight = 1.0 / class_counts[class_index]
    sample_weights[train_labels == class_index] = class_weight

sample_weights = torch.as_tensor(
    sample_weights,
    dtype=torch.double
)

vgg16_sampler = WeightedRandomSampler(
    weights=sample_weights,
    num_samples=len(train_labels),
    replacement=True
)

vgg16_train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    sampler=vgg16_sampler,
    num_workers=NUM_WORKERS,
    pin_memory=True
)

print("VGG16 balanced training loader created.")
print("-" * 60)
print(f"Original training samples: {len(train_dataset)}")
print(f"Non-melanoma samples: {class_counts[0]}")
print(f"Melanoma samples: {class_counts[1]}")
print("Sampling method: WeightedRandomSampler")
print("Replacement: True")
print(f"Samples per epoch: {len(vgg16_train_loader.dataset)}")

VGG16 balanced training loader created.
------------------------------------------------------------
Original training samples: 6981
Non-melanoma samples: 6208
Melanoma samples: 773
Sampling method: WeightedRandomSampler
Replacement: True
Samples per epoch: 6981


In [72]:
# --- Train the VGG16 benchmark model ---

import time
import copy

vgg16_best_f1 = -1.0
vgg16_best_epoch = 0
vgg16_history = []
vgg16_patience_counter = 0

VGG16_CHECKPOINT_PATH = "/kaggle/working/ham10000_vgg16_balanced_best.pth"

training_start = time.time()

for epoch in range(1, MAX_EPOCHS + 1):

    epoch_start = time.time()

    vgg16_model.train()

    running_loss = 0.0
    training_samples = 0

    for images, labels in vgg16_train_loader:

        images = images.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)

        vgg16_optimizer.zero_grad()

        outputs = vgg16_model(images)
        loss = vgg16_criterion(outputs, labels)

        loss.backward()
        vgg16_optimizer.step()

        batch_size = images.size(0)

        running_loss += loss.item() * batch_size
        training_samples += batch_size

    train_loss = running_loss / training_samples

    val_metrics = evaluate_model(
        vgg16_model,
        val_loader,
        vgg16_criterion,
        DEVICE
    )

    epoch_time = time.time() - epoch_start

    vgg16_history.append({
        "epoch": epoch,
        "train_loss": train_loss,
        **val_metrics,
        "epoch_time_seconds": epoch_time
    })

    current_f1 = val_metrics["f1"]

    print(
        f"Epoch {epoch:02d}/{MAX_EPOCHS} | "
        f"Train Loss {train_loss:.4f} | "
        f"Val Loss {val_metrics['loss']:.4f} | "
        f"Val Acc {val_metrics['accuracy']:.4f} | "
        f"Val F1 {val_metrics['f1']:.4f} | "
        f"Val AUC {val_metrics['roc_auc']:.4f} | "
        f"Time {epoch_time:.1f}s"
    )

    if current_f1 > vgg16_best_f1:

        vgg16_best_f1 = current_f1
        vgg16_best_epoch = epoch
        vgg16_patience_counter = 0

        torch.save(
            {
                "epoch": epoch,
                "model_state_dict": vgg16_model.state_dict(),
                "optimizer_state_dict": vgg16_optimizer.state_dict(),
                "best_val_f1": vgg16_best_f1,
                "history": vgg16_history,
            },
            VGG16_CHECKPOINT_PATH
        )

        print(f"  Best checkpoint saved at epoch {epoch}.")

    else:

        vgg16_patience_counter += 1

        if vgg16_patience_counter >= EARLY_STOPPING_PATIENCE:

            print(
                f"Early stopping triggered after "
                f"{EARLY_STOPPING_PATIENCE} epochs without improvement."
            )

            break

total_training_time = time.time() - training_start

print("-" * 60)
print("VGG16 training completed.")
print(f"Best epoch: {vgg16_best_epoch}")
print(f"Best validation F1: {vgg16_best_f1:.4f}")
print(f"Total training time: {total_training_time / 60:.2f} minutes")
print(f"Checkpoint: {VGG16_CHECKPOINT_PATH}")

Epoch 01/20 | Train Loss 0.1182 | Val Loss 0.4656 | Val Acc 0.8244 | Val F1 0.5171 | Val AUC 0.9081 | Time 91.0s
  Best checkpoint saved at epoch 1.
Epoch 02/20 | Train Loss 0.0955 | Val Loss 0.3857 | Val Acc 0.8701 | Val F1 0.5757 | Val AUC 0.9187 | Time 91.0s
  Best checkpoint saved at epoch 2.
Epoch 03/20 | Train Loss 0.1031 | Val Loss 0.4337 | Val Acc 0.8708 | Val F1 0.5769 | Val AUC 0.9138 | Time 91.1s
  Best checkpoint saved at epoch 3.
Epoch 04/20 | Train Loss 0.0924 | Val Loss 0.4242 | Val Acc 0.8675 | Val F1 0.5634 | Val AUC 0.9128 | Time 91.2s
Epoch 05/20 | Train Loss 0.0914 | Val Loss 0.5216 | Val Acc 0.8747 | Val F1 0.5596 | Val AUC 0.9019 | Time 90.7s
Epoch 06/20 | Train Loss 0.0865 | Val Loss 0.3894 | Val Acc 0.8708 | Val F1 0.5787 | Val AUC 0.9189 | Time 90.6s
  Best checkpoint saved at epoch 6.
Epoch 07/20 | Train Loss 0.0886 | Val Loss 0.4677 | Val Acc 0.8492 | Val F1 0.5462 | Val AUC 0.9032 | Time 90.8s
Epoch 08/20 | Train Loss 0.0856 | Val Loss 0.3775 | Val Acc 0.888

In [73]:
# --- Reload the best VGG16 checkpoint before final evaluation ---

checkpoint = torch.load(
    VGG16_CHECKPOINT_PATH,
    map_location=DEVICE,
    weights_only=False
)

vgg16_model.load_state_dict(checkpoint["model_state_dict"])

vgg16_best_epoch = checkpoint["epoch"]
vgg16_best_f1 = checkpoint["best_val_f1"]

print("Best VGG16 checkpoint reloaded successfully.")
print("-" * 60)
print(f"Best epoch: {vgg16_best_epoch}")
print(f"Best validation F1: {vgg16_best_f1:.4f}")
print(f"Checkpoint path: {VGG16_CHECKPOINT_PATH}")

Best VGG16 checkpoint reloaded successfully.
------------------------------------------------------------
Best epoch: 12
Best validation F1: 0.5962
Checkpoint path: /kaggle/working/ham10000_vgg16_balanced_best.pth


In [74]:
# --- Perform the official untouched HAM10000 test evaluation for VGG16 ---

import time

vgg16_test_start = time.time()

vgg16_test_metrics = evaluate_model(
    vgg16_model,
    test_loader,
    vgg16_criterion,
    DEVICE
)

vgg16_test_time = time.time() - vgg16_test_start

print("VGG16 official test evaluation completed.")
print("-" * 60)
print(f"Test Loss: {vgg16_test_metrics['loss']:.6f}")
print(f"Accuracy: {vgg16_test_metrics['accuracy']:.6f}")
print(f"Precision: {vgg16_test_metrics['precision']:.6f}")
print(f"Recall: {vgg16_test_metrics['recall']:.6f}")
print(f"F1 Score: {vgg16_test_metrics['f1']:.6f}")
print(f"ROC-AUC: {vgg16_test_metrics['roc_auc']:.6f}")
print(f"Confusion Matrix:")
print(vgg16_test_metrics["confusion_matrix"])
print(f"Evaluation time: {vgg16_test_time / 60:.2f} minutes")
print(
    f"Average inference time: "
    f"{vgg16_test_time / len(test_dataset):.4f} seconds/image"
)

VGG16 official test evaluation completed.
------------------------------------------------------------
Test Loss: 0.353574
Accuracy: 0.879494
Precision: 0.469027
Recall: 0.634731
F1 Score: 0.539440
ROC-AUC: 0.887788
Confusion Matrix:
[[1215  120]
 [  61  106]]
Evaluation time: 0.11 minutes
Average inference time: 0.0044 seconds/image


In [75]:
# --- Save the VGG16 benchmark results and training history ---

import json
import torch

VGG16_RESULTS_PATH = "/kaggle/working/ham10000_vgg16_balanced_results.json"
VGG16_HISTORY_PATH = "/kaggle/working/ham10000_vgg16_balanced_training_history.pt"

vgg16_results = {
    "model": "VGG16",
    "dataset": "HAM10000",
    "task": "Binary melanoma classification",
    "best_epoch": int(vgg16_best_epoch),
    "best_validation_f1": float(vgg16_best_f1),
    "test_loss": float(vgg16_test_metrics["loss"]),
    "test_accuracy": float(vgg16_test_metrics["accuracy"]),
    "test_precision": float(vgg16_test_metrics["precision"]),
    "test_recall": float(vgg16_test_metrics["recall"]),
    "test_f1": float(vgg16_test_metrics["f1"]),
    "test_roc_auc": float(vgg16_test_metrics["roc_auc"]),
    "confusion_matrix": vgg16_test_metrics["confusion_matrix"].tolist(),
    "test_evaluation_time_seconds": float(vgg16_test_time),
    "average_inference_time_seconds": float(
        vgg16_test_time / len(test_dataset)
    ),
    "sampling": "WeightedRandomSampler",
    "pretrained": "ImageNet",
    "input_size": "224x224",
    "batch_size": int(BATCH_SIZE),
    "learning_rate": float(LEARNING_RATE),
    "weight_decay": float(WEIGHT_DECAY),
    "max_epochs": int(MAX_EPOCHS),
    "early_stopping_patience": int(EARLY_STOPPING_PATIENCE)
}

with open(VGG16_RESULTS_PATH, "w") as f:
    json.dump(vgg16_results, f, indent=2)

torch.save(
    vgg16_history,
    VGG16_HISTORY_PATH
)

print("VGG16 experiment artifacts saved.")
print("-" * 60)
print(f"Results: {VGG16_RESULTS_PATH}")
print(f"Training history: {VGG16_HISTORY_PATH}")
print(f"Best checkpoint: {VGG16_CHECKPOINT_PATH}")

VGG16 experiment artifacts saved.
------------------------------------------------------------
Results: /kaggle/working/ham10000_vgg16_balanced_results.json
Training history: /kaggle/working/ham10000_vgg16_balanced_training_history.pt
Best checkpoint: /kaggle/working/ham10000_vgg16_balanced_best.pth


In [76]:
# --- Create a complete VGG16 benchmark backup ---

import zipfile
from pathlib import Path

vgg16_backup_path = Path(
    "/kaggle/working/HAM10000_VGG16_Balanced_Baseline_Backup.zip"
)

vgg16_files = [
    Path("/kaggle/working/ham10000_vgg16_balanced_best.pth"),
    Path("/kaggle/working/ham10000_vgg16_balanced_training_history.pt"),
    Path("/kaggle/working/ham10000_vgg16_balanced_results.json"),
]

with zipfile.ZipFile(
    vgg16_backup_path,
    "w",
    compression=zipfile.ZIP_DEFLATED
) as zip_file:

    for file_path in vgg16_files:
        zip_file.write(
            file_path,
            arcname=file_path.name
        )

print("VGG16 backup created successfully.")
print("-" * 60)

for file_path in vgg16_files:
    size_mb = file_path.stat().st_size / (1024 ** 2)
    print(f"{file_path.name} | {size_mb:.2f} MB")

backup_size_mb = vgg16_backup_path.stat().st_size / (1024 ** 2)

print("-" * 60)
print(f"Backup ZIP: {vgg16_backup_path}")
print(f"Backup size: {backup_size_mb:.2f} MB")

VGG16 backup created successfully.
------------------------------------------------------------
ham10000_vgg16_balanced_best.pth | 1536.63 MB
ham10000_vgg16_balanced_training_history.pt | 0.01 MB
ham10000_vgg16_balanced_results.json | 0.00 MB
------------------------------------------------------------
Backup ZIP: /kaggle/working/HAM10000_VGG16_Balanced_Baseline_Backup.zip
Backup size: 1398.80 MB


In [1]:
# --- Verify that the VGG16 backup ZIP is available ---

from pathlib import Path

vgg16_backup = Path(
    "/kaggle/working/HAM10000_VGG16_Balanced_Baseline_Backup.zip"
)

if vgg16_backup.exists():
    size_mb = vgg16_backup.stat().st_size / (1024 ** 2)

    print("VGG16 backup ZIP found.")
    print("-" * 60)
    print(f"File: {vgg16_backup.name}")
    print(f"Size: {size_mb:.2f} MB")
    print(f"Path: {vgg16_backup}")
else:
    print("ERROR: VGG16 backup ZIP was not found.")

ERROR: VGG16 backup ZIP was not found.


In [78]:
# --- Split the VGG16 backup into smaller downloadable parts ---

from pathlib import Path

backup_source = Path(
    "/kaggle/working/HAM10000_VGG16_Balanced_Baseline_Backup.zip"
)

part_size = 100 * 1024 * 1024

with open(backup_source, "rb") as source_file:
    part_number = 1

    while True:
        data = source_file.read(part_size)

        if not data:
            break

        part_path = Path(
            f"/kaggle/working/"
            f"HAM10000_VGG16_Balanced_Baseline_Backup.part{part_number:02d}"
        )

        with open(part_path, "wb") as part_file:
            part_file.write(data)

        print(
            f"Created {part_path.name} | "
            f"{len(data) / (1024 ** 2):.2f} MB"
        )

        part_number += 1

print("-" * 60)
print(f"Total parts created: {part_number - 1}")

Created HAM10000_VGG16_Balanced_Baseline_Backup.part01 | 100.00 MB
Created HAM10000_VGG16_Balanced_Baseline_Backup.part02 | 100.00 MB
Created HAM10000_VGG16_Balanced_Baseline_Backup.part03 | 100.00 MB
Created HAM10000_VGG16_Balanced_Baseline_Backup.part04 | 100.00 MB
Created HAM10000_VGG16_Balanced_Baseline_Backup.part05 | 100.00 MB
Created HAM10000_VGG16_Balanced_Baseline_Backup.part06 | 100.00 MB
Created HAM10000_VGG16_Balanced_Baseline_Backup.part07 | 100.00 MB
Created HAM10000_VGG16_Balanced_Baseline_Backup.part08 | 100.00 MB
Created HAM10000_VGG16_Balanced_Baseline_Backup.part09 | 100.00 MB
Created HAM10000_VGG16_Balanced_Baseline_Backup.part10 | 100.00 MB
Created HAM10000_VGG16_Balanced_Baseline_Backup.part11 | 100.00 MB
Created HAM10000_VGG16_Balanced_Baseline_Backup.part12 | 100.00 MB
Created HAM10000_VGG16_Balanced_Baseline_Backup.part13 | 100.00 MB
Created HAM10000_VGG16_Balanced_Baseline_Backup.part14 | 98.80 MB
------------------------------------------------------------
To

In [79]:
# --- Verify the complete VGG16 research artifacts ---

from pathlib import Path

vgg16_files = [
    "/kaggle/working/ham10000_vgg16_balanced_best.pth",
    "/kaggle/working/ham10000_vgg16_balanced_training_history.pt",
    "/kaggle/working/ham10000_vgg16_balanced_results.json",
    "/kaggle/working/HAM10000_VGG16_Balanced_Baseline_Backup.zip"
]

print("VGG16 artifact verification")
print("-" * 60)

for file_path in vgg16_files:
    path = Path(file_path)

    if path.exists():
        size_mb = path.stat().st_size / (1024 ** 2)
        print(f"FOUND | {path.name} | {size_mb:.2f} MB")
    else:
        print(f"MISSING | {path.name}")

print("-" * 60)
print("Verification complete.")

VGG16 artifact verification
------------------------------------------------------------
FOUND | ham10000_vgg16_balanced_best.pth | 1536.63 MB
FOUND | ham10000_vgg16_balanced_training_history.pt | 0.01 MB
FOUND | ham10000_vgg16_balanced_results.json | 0.00 MB
FOUND | HAM10000_VGG16_Balanced_Baseline_Backup.zip | 1398.80 MB
------------------------------------------------------------
Verification complete.


In [6]:
# Purpose: Import required libraries and define the common HAM10000 benchmark configuration.

import os
import json
import random
import time
import copy
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

import torchvision
from torchvision import models, transforms

from PIL import Image

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix
)

SEED = 42
NUM_CLASSES = 2
BATCH_SIZE = 32
NUM_WORKERS = 2

LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-5

MAX_EPOCHS = 20
EARLY_STOPPING_PATIENCE = 5

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("PyTorch:", torch.__version__)
print("Torchvision:", torchvision.__version__)
print("Device:", DEVICE)
print("GPU count:", torch.cuda.device_count())

PyTorch: 2.10.0+cu128
Torchvision: 0.25.0+cu128
Device: cuda
GPU count: 2


In [7]:
# Purpose: Set deterministic random seeds for reproducible VGG19 training.

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


set_seed(SEED)

print("Seed:", SEED)
print("PyTorch seed:", torch.initial_seed())
print("CUDA deterministic:", torch.backends.cudnn.deterministic)
print("CUDA benchmark:", torch.backends.cudnn.benchmark)

Seed: 42
PyTorch seed: 42
CUDA deterministic: True
CUDA benchmark: False


In [8]:
# Purpose: Locate and verify the frozen HAM10000 dataset supplied to the Kaggle notebook.

DATASET_ROOT = "/kaggle/input/datasets/akansha1009/ham10000-preprocessed-segmentation"

TRAIN_DIR = os.path.join(DATASET_ROOT, "train")
VAL_DIR = os.path.join(DATASET_ROOT, "validation")
TEST_DIR = os.path.join(DATASET_ROOT, "test")

print("Dataset root:", DATASET_ROOT)
print("Train exists:", os.path.exists(TRAIN_DIR))
print("Validation exists:", os.path.exists(VAL_DIR))
print("Test exists:", os.path.exists(TEST_DIR))

print("\nTrain files:", os.listdir(TRAIN_DIR))
print("Validation files:", os.listdir(VAL_DIR))
print("Test files:", os.listdir(TEST_DIR))

Dataset root: /kaggle/input/datasets/akansha1009/ham10000-preprocessed-segmentation
Train exists: True
Validation exists: True
Test exists: True

Train files: ['images.npy', 'masks.npy', 'metadata.csv']
Validation files: ['images.npy', 'masks.npy', 'metadata.csv']
Test files: ['images.npy', 'masks.npy', 'metadata.csv']


In [9]:
# Purpose: Load the frozen HAM10000 arrays and verify their shapes and data types.

train_images = np.load(os.path.join(TRAIN_DIR, "images.npy"), mmap_mode="r")
train_masks = np.load(os.path.join(TRAIN_DIR, "masks.npy"), mmap_mode="r")
train_metadata = pd.read_csv(os.path.join(TRAIN_DIR, "metadata.csv"))

val_images = np.load(os.path.join(VAL_DIR, "images.npy"), mmap_mode="r")
val_masks = np.load(os.path.join(VAL_DIR, "masks.npy"), mmap_mode="r")
val_metadata = pd.read_csv(os.path.join(VAL_DIR, "metadata.csv"))

test_images = np.load(os.path.join(TEST_DIR, "images.npy"), mmap_mode="r")
test_masks = np.load(os.path.join(TEST_DIR, "masks.npy"), mmap_mode="r")
test_metadata = pd.read_csv(os.path.join(TEST_DIR, "metadata.csv"))

print("Train images:", train_images.shape, train_images.dtype)
print("Train masks:", train_masks.shape, train_masks.dtype)
print("Train metadata:", train_metadata.shape)

print("\nValidation images:", val_images.shape, val_images.dtype)
print("Validation masks:", val_masks.shape, val_masks.dtype)
print("Validation metadata:", val_metadata.shape)

print("\nTest images:", test_images.shape, test_images.dtype)
print("Test masks:", test_masks.shape, test_masks.dtype)
print("Test metadata:", test_metadata.shape)

print("\nMetadata columns:", list(train_metadata.columns))

Train images: (6981, 224, 224, 3) float32
Train masks: (6981, 224, 224) uint8
Train metadata: (6981, 2)

Validation images: (1532, 224, 224, 3) float32
Validation masks: (1532, 224, 224) uint8
Validation metadata: (1532, 2)

Test images: (1502, 224, 224, 3) float32
Test masks: (1502, 224, 224) uint8
Test metadata: (1502, 2)

Metadata columns: ['image_id', 'dx']


In [10]:
# Purpose: Verify the frozen binary melanoma versus non-melanoma class distribution.

print("Training class distribution:")
print(train_metadata["dx"].value_counts().sort_index())

print("\nValidation class distribution:")
print(val_metadata["dx"].value_counts().sort_index())

print("\nTest class distribution:")
print(test_metadata["dx"].value_counts().sort_index())

print("\nBinary melanoma counts:")

print("Train melanoma:", (train_metadata["dx"] == "mel").sum())
print("Train non-melanoma:", (train_metadata["dx"] != "mel").sum())

print("Validation melanoma:", (val_metadata["dx"] == "mel").sum())
print("Validation non-melanoma:", (val_metadata["dx"] != "mel").sum())

print("Test melanoma:", (test_metadata["dx"] == "mel").sum())
print("Test non-melanoma:", (test_metadata["dx"] != "mel").sum())

Training class distribution:
dx
akiec     222
bcc       361
bkl       772
df         71
mel       773
nv       4683
vasc       99
Name: count, dtype: int64

Validation class distribution:
dx
akiec      53
bcc        82
bkl       160
df         24
mel       173
nv       1018
vasc       22
Name: count, dtype: int64

Test class distribution:
dx
akiec      52
bcc        71
bkl       167
df         20
mel       167
nv       1004
vasc       21
Name: count, dtype: int64

Binary melanoma counts:
Train melanoma: 773
Train non-melanoma: 6208
Validation melanoma: 173
Validation non-melanoma: 1359
Test melanoma: 167
Test non-melanoma: 1335


In [11]:
# Purpose: Define the HAM10000 classification dataset and consistent ImageNet preprocessing.

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.2),
    transforms.RandomRotation(degrees=15),
    transforms.ColorJitter(
        brightness=0.15,
        contrast=0.15,
        saturation=0.10,
        hue=0.02
    ),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD)
])

eval_transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD)
])


class HAM10000ClassificationDataset(Dataset):
    def __init__(self, images, metadata, transform=None):
        self.images = images
        self.metadata = metadata.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.metadata)

    def __getitem__(self, index):
        image = self.images[index]
        diagnosis = self.metadata.iloc[index]["dx"]

        label = 1 if diagnosis == "mel" else 0

        if self.transform:
            image = self.transform(image)

        return image, torch.tensor(label, dtype=torch.long)


train_dataset = HAM10000ClassificationDataset(
    train_images,
    train_metadata,
    transform=train_transform
)

val_dataset = HAM10000ClassificationDataset(
    val_images,
    val_metadata,
    transform=eval_transform
)

test_dataset = HAM10000ClassificationDataset(
    test_images,
    test_metadata,
    transform=eval_transform
)

print("Train dataset:", len(train_dataset))
print("Validation dataset:", len(val_dataset))
print("Test dataset:", len(test_dataset))

Train dataset: 6981
Validation dataset: 1532
Test dataset: 1502


In [12]:
# Purpose: Verify that an individual training sample has the correct tensor shape and binary label.

sample_image, sample_label = train_dataset[0]

print("Image shape:", sample_image.shape)
print("Image dtype:", sample_image.dtype)
print("Label:", sample_label.item())
print("Label dtype:", sample_label.dtype)

print("\nImage range after normalization:")
print("Minimum:", sample_image.min().item())
print("Maximum:", sample_image.max().item())

Image shape: torch.Size([3, 224, 224])
Image dtype: torch.float32
Label: 0
Label dtype: torch.int64

Image range after normalization:
Minimum: -2.1179039478302
Maximum: 2.3088455200195312


In [13]:
# Purpose: Create a balanced training sampler so melanoma and non-melanoma contribute more evenly during training.

train_labels = np.array([
    1 if diagnosis == "mel" else 0
    for diagnosis in train_metadata["dx"]
])

class_counts = np.bincount(train_labels)
class_weights = 1.0 / class_counts

sample_weights = class_weights[train_labels]
sample_weights = torch.as_tensor(sample_weights, dtype=torch.double)

balanced_sampler = WeightedRandomSampler(
    weights=sample_weights,
    num_samples=len(train_labels),
    replacement=True
)

vgg19_train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    sampler=balanced_sampler,
    num_workers=NUM_WORKERS,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True
)

print("Original class counts:")
print("Non-melanoma:", class_counts[0])
print("Melanoma:", class_counts[1])

print("\nBalanced samples per epoch:", len(vgg19_train_loader.dataset))
print("Training batches:", len(vgg19_train_loader))
print("Validation batches:", len(val_loader))
print("Test batches:", len(test_loader))
print("Replacement sampling:", balanced_sampler.replacement)

Original class counts:
Non-melanoma: 6208
Melanoma: 773

Balanced samples per epoch: 6981
Training batches: 219
Validation batches: 48
Test batches: 47
Replacement sampling: True


In [14]:
# Purpose: Create the ImageNet-pretrained VGG19 model for binary melanoma classification.

vgg19_model = models.vgg19(weights=models.VGG19_Weights.DEFAULT)

vgg19_model.classifier[6] = nn.Linear(
    vgg19_model.classifier[6].in_features,
    NUM_CLASSES
)

vgg19_model = vgg19_model.to(DEVICE)

if torch.cuda.device_count() > 1:
    vgg19_model = nn.DataParallel(vgg19_model)

total_params = sum(
    parameter.numel()
    for parameter in vgg19_model.parameters()
)

trainable_params = sum(
    parameter.numel()
    for parameter in vgg19_model.parameters()
    if parameter.requires_grad
)

print("Model: VGG19")
print("Total parameters:", total_params)
print("Trainable parameters:", trainable_params)
print("Device:", DEVICE)
print("GPU count:", torch.cuda.device_count())

Downloading: "https://download.pytorch.org/models/vgg19-dcbb9e9d.pth" to /root/.cache/torch/hub/checkpoints/vgg19-dcbb9e9d.pth


100%|██████████| 548M/548M [00:02<00:00, 206MB/s]  


Model: VGG19
Total parameters: 139578434
Trainable parameters: 139578434
Device: cuda
GPU count: 2


In [15]:
# Purpose: Configure the VGG19 loss function and optimizer for balanced binary classification.

vgg19_criterion = nn.CrossEntropyLoss()

vgg19_optimizer = optim.Adam(
    vgg19_model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY
)

print("Loss function:", vgg19_criterion)
print("Optimizer:", vgg19_optimizer.__class__.__name__)
print("Learning rate:", LEARNING_RATE)
print("Weight decay:", WEIGHT_DECAY)

Loss function: CrossEntropyLoss()
Optimizer: Adam
Learning rate: 0.0001
Weight decay: 1e-05


In [16]:
# Purpose: Train VGG19 with balanced sampling, validation monitoring, early stopping, and best-checkpoint saving.

vgg19_best_path = "/kaggle/working/ham10000_vgg19_balanced_best.pth"
vgg19_history_path = "/kaggle/working/ham10000_vgg19_balanced_training_history.pt"

vgg19_history = {
    "train_loss": [],
    "val_loss": [],
    "val_accuracy": [],
    "val_precision": [],
    "val_recall": [],
    "val_f1": [],
    "val_auc": []
}

best_val_f1 = -1.0
best_epoch = 0
epochs_without_improvement = 0

for epoch in range(1, MAX_EPOCHS + 1):
    epoch_start = time.time()

    vgg19_model.train()
    running_train_loss = 0.0
    train_samples = 0

    for images, labels in vgg19_train_loader:
        images = images.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)

        vgg19_optimizer.zero_grad(set_to_none=True)

        outputs = vgg19_model(images)
        loss = vgg19_criterion(outputs, labels)

        loss.backward()
        vgg19_optimizer.step()

        batch_size = labels.size(0)
        running_train_loss += loss.item() * batch_size
        train_samples += batch_size

    train_loss = running_train_loss / train_samples

    vgg19_model.eval()

    running_val_loss = 0.0
    val_samples = 0
    all_val_labels = []
    all_val_predictions = []
    all_val_probabilities = []

    with torch.no_grad():
        for images, labels in val_loader:
            images = images.to(DEVICE, non_blocking=True)
            labels = labels.to(DEVICE, non_blocking=True)

            outputs = vgg19_model(images)
            loss = vgg19_criterion(outputs, labels)

            probabilities = torch.softmax(outputs, dim=1)[:, 1]
            predictions = torch.argmax(outputs, dim=1)

            batch_size = labels.size(0)
            running_val_loss += loss.item() * batch_size
            val_samples += batch_size

            all_val_labels.extend(labels.cpu().numpy())
            all_val_predictions.extend(predictions.cpu().numpy())
            all_val_probabilities.extend(probabilities.cpu().numpy())

    val_loss = running_val_loss / val_samples

    val_accuracy = accuracy_score(
        all_val_labels,
        all_val_predictions
    )

    val_precision = precision_score(
        all_val_labels,
        all_val_predictions,
        zero_division=0
    )

    val_recall = recall_score(
        all_val_labels,
        all_val_predictions,
        zero_division=0
    )

    val_f1 = f1_score(
        all_val_labels,
        all_val_predictions,
        zero_division=0
    )

    val_auc = roc_auc_score(
        all_val_labels,
        all_val_probabilities
    )

    vgg19_history["train_loss"].append(train_loss)
    vgg19_history["val_loss"].append(val_loss)
    vgg19_history["val_accuracy"].append(val_accuracy)
    vgg19_history["val_precision"].append(val_precision)
    vgg19_history["val_recall"].append(val_recall)
    vgg19_history["val_f1"].append(val_f1)
    vgg19_history["val_auc"].append(val_auc)

    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        best_epoch = epoch
        epochs_without_improvement = 0

        torch.save(
            {
                "epoch": epoch,
                "model_state_dict": vgg19_model.state_dict(),
                "optimizer_state_dict": vgg19_optimizer.state_dict(),
                "best_val_f1": best_val_f1,
                "history": vgg19_history
            },
            vgg19_best_path
        )

        best_marker = "  <-- BEST"
    else:
        epochs_without_improvement += 1
        best_marker = ""

    elapsed = time.time() - epoch_start

    print(
        f"Epoch {epoch:02d}/{MAX_EPOCHS} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Val Loss: {val_loss:.4f} | "
        f"Val Acc: {val_accuracy:.4f} | "
        f"Val Precision: {val_precision:.4f} | "
        f"Val Recall: {val_recall:.4f} | "
        f"Val F1: {val_f1:.4f} | "
        f"Val AUC: {val_auc:.4f} | "
        f"Time: {elapsed:.1f}s"
        f"{best_marker}"
    )

    torch.save(vgg19_history, vgg19_history_path)

    if epochs_without_improvement >= EARLY_STOPPING_PATIENCE:
        print(
            f"\nEarly stopping triggered after {epoch} epochs."
        )
        break

print("\nVGG19 training complete.")
print("Best epoch:", best_epoch)
print("Best validation F1:", f"{best_val_f1:.4f}")
print("Best model saved to:", vgg19_best_path)
print("Training history saved to:", vgg19_history_path)

Epoch 01/20 | Train Loss: 0.5201 | Val Loss: 0.6429 | Val Acc: 0.5783 | Val Precision: 0.2098 | Val Recall: 0.9884 | Val F1: 0.3462 | Val AUC: 0.9129 | Time: 103.7s  <-- BEST
Epoch 02/20 | Train Loss: 0.4298 | Val Loss: 0.2338 | Val Acc: 0.9178 | Val Precision: 0.6298 | Val Recall: 0.6590 | Val F1: 0.6441 | Val AUC: 0.9236 | Time: 106.2s  <-- BEST
Epoch 03/20 | Train Loss: 0.3983 | Val Loss: 0.6574 | Val Acc: 0.6475 | Val Precision: 0.2390 | Val Recall: 0.9711 | Val F1: 0.3836 | Val AUC: 0.9145 | Time: 101.4s
Epoch 04/20 | Train Loss: 0.3479 | Val Loss: 0.2502 | Val Acc: 0.8910 | Val Precision: 0.5130 | Val Recall: 0.6821 | Val F1: 0.5856 | Val AUC: 0.9192 | Time: 101.2s
Epoch 05/20 | Train Loss: 0.3480 | Val Loss: 0.9705 | Val Acc: 0.6338 | Val Precision: 0.2320 | Val Recall: 0.9711 | Val F1: 0.3746 | Val AUC: 0.9206 | Time: 101.2s
Epoch 06/20 | Train Loss: 0.2928 | Val Loss: 0.3127 | Val Acc: 0.8662 | Val Precision: 0.4477 | Val Recall: 0.7919 | Val F1: 0.5720 | Val AUC: 0.9252 | Tim

In [17]:
# Purpose: Load the best VGG19 checkpoint and evaluate it on the untouched HAM10000 test set.

checkpoint = torch.load(
    vgg19_best_path,
    map_location=DEVICE,
    weights_only=False
)

vgg19_model.load_state_dict(checkpoint["model_state_dict"])

print("Loaded best checkpoint from epoch:", checkpoint["epoch"])
print("Best validation F1:", checkpoint["best_val_f1"])

vgg19_model.eval()

test_loss = 0.0
test_samples = 0

all_test_labels = []
all_test_predictions = []
all_test_probabilities = []

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)

        outputs = vgg19_model(images)
        loss = vgg19_criterion(outputs, labels)

        probabilities = torch.softmax(outputs, dim=1)[:, 1]
        predictions = torch.argmax(outputs, dim=1)

        batch_size = labels.size(0)

        test_loss += loss.item() * batch_size
        test_samples += batch_size

        all_test_labels.extend(labels.cpu().numpy())
        all_test_predictions.extend(predictions.cpu().numpy())
        all_test_probabilities.extend(probabilities.cpu().numpy())

test_loss = test_loss / test_samples

test_accuracy = accuracy_score(
    all_test_labels,
    all_test_predictions
)

test_precision = precision_score(
    all_test_labels,
    all_test_predictions,
    zero_division=0
)

test_recall = recall_score(
    all_test_labels,
    all_test_predictions,
    zero_division=0
)

test_f1 = f1_score(
    all_test_labels,
    all_test_predictions,
    zero_division=0
)

test_auc = roc_auc_score(
    all_test_labels,
    all_test_probabilities
)

test_cm = confusion_matrix(
    all_test_labels,
    all_test_predictions
)

print("\nVGG19 Official Test Results")
print("-" * 40)
print(f"Test Loss:     {test_loss:.6f}")
print(f"Accuracy:      {test_accuracy:.6f}")
print(f"Precision:     {test_precision:.6f}")
print(f"Recall:        {test_recall:.6f}")
print(f"F1 Score:      {test_f1:.6f}")
print(f"ROC-AUC:       {test_auc:.6f}")

print("\nConfusion Matrix:")
print(test_cm)

Loaded best checkpoint from epoch: 2
Best validation F1: 0.6440677966101694

VGG19 Official Test Results
----------------------------------------
Test Loss:     0.269522
Accuracy:      0.903462
Precision:     0.568750
Recall:        0.544910
F1 Score:      0.556575
ROC-AUC:       0.872484

Confusion Matrix:
[[1266   69]
 [  76   91]]


In [18]:
# Purpose: Save the official VGG19 test results as a reproducible JSON artifact.

vgg19_results = {
    "model": "VGG19",
    "dataset": "HAM10000",
    "task": "Binary melanoma vs non-melanoma classification",
    "best_epoch": int(checkpoint["epoch"]),
    "best_validation_f1": float(checkpoint["best_val_f1"]),
    "test_loss": float(test_loss),
    "test_accuracy": float(test_accuracy),
    "test_precision": float(test_precision),
    "test_recall": float(test_recall),
    "test_f1": float(test_f1),
    "test_roc_auc": float(test_auc),
    "confusion_matrix": test_cm.tolist(),
    "seed": SEED,
    "batch_size": BATCH_SIZE,
    "learning_rate": LEARNING_RATE,
    "weight_decay": WEIGHT_DECAY,
    "max_epochs": MAX_EPOCHS,
    "early_stopping_patience": EARLY_STOPPING_PATIENCE,
    "training_strategy": "WeightedRandomSampler with ordinary CrossEntropyLoss"
}

vgg19_results_path = "/kaggle/working/ham10000_vgg19_balanced_results.json"

with open(vgg19_results_path, "w") as f:
    json.dump(vgg19_results, f, indent=4)

print("VGG19 results saved successfully.")
print("Path:", vgg19_results_path)

print("\nSaved test F1:", vgg19_results["test_f1"])
print("Saved test ROC-AUC:", vgg19_results["test_roc_auc"])

VGG19 results saved successfully.
Path: /kaggle/working/ham10000_vgg19_balanced_results.json

Saved test F1: 0.5565749235474006
Saved test ROC-AUC: 0.8724842449931597


In [19]:
# Purpose: Package all essential VGG19 benchmark artifacts into one backup archive.

import zipfile
import os

vgg19_backup_path = "/kaggle/working/HAM10000_VGG19_Balanced_Baseline_Backup.zip"

vgg19_files_to_backup = [
    vgg19_best_path,
    vgg19_history_path,
    vgg19_results_path
]

with zipfile.ZipFile(
    vgg19_backup_path,
    mode="w",
    compression=zipfile.ZIP_DEFLATED
) as zip_file:

    for file_path in vgg19_files_to_backup:
        zip_file.write(
            file_path,
            arcname=os.path.basename(file_path)
        )

print("VGG19 backup created successfully.")
print("Backup path:", vgg19_backup_path)

backup_size_mb = os.path.getsize(vgg19_backup_path) / (1024 ** 2)

print(f"Backup size: {backup_size_mb:.2f} MB")

print("\nFiles included:")
for file_path in vgg19_files_to_backup:
    file_size_mb = os.path.getsize(file_path) / (1024 ** 2)
    print(f"{os.path.basename(file_path)} | {file_size_mb:.2f} MB")

VGG19 backup created successfully.
Backup path: /kaggle/working/HAM10000_VGG19_Balanced_Baseline_Backup.zip
Backup size: 1518.43 MB

Files included:
ham10000_vgg19_balanced_best.pth | 1597.40 MB
ham10000_vgg19_balanced_training_history.pt | 0.00 MB
ham10000_vgg19_balanced_results.json | 0.00 MB


In [20]:
# Purpose: Verify that the VGG19 backup archive contains all required benchmark artifacts.

with zipfile.ZipFile(vgg19_backup_path, "r") as zip_file:
    backup_contents = zip_file.namelist()

print("Backup contents:")

for file_name in backup_contents:
    print(" -", file_name)

required_files = [
    "ham10000_vgg19_balanced_best.pth",
    "ham10000_vgg19_balanced_training_history.pt",
    "ham10000_vgg19_balanced_results.json"
]

missing_files = [
    file_name
    for file_name in required_files
    if file_name not in backup_contents
]

print("\nMissing required files:", missing_files)

if not missing_files:
    print("Backup integrity check: PASSED")
else:
    print("Backup integrity check: FAILED")

Backup contents:
 - ham10000_vgg19_balanced_best.pth
 - ham10000_vgg19_balanced_training_history.pt
 - ham10000_vgg19_balanced_results.json

Missing required files: []
Backup integrity check: PASSED


In [21]:
# Verify that the existing VGG16 backup is still available

import os

backup_path = "/kaggle/working/HAM10000_VGG16_Balanced_Baseline_Backup.zip"

if os.path.exists(backup_path):
    size_gb = os.path.getsize(backup_path) / (1024 ** 3)
    print("VGG16 backup FOUND")
    print(f"Path: {backup_path}")
    print(f"Size: {size_gb:.2f} GB")
else:
    print("VGG16 backup NOT FOUND")

VGG16 backup NOT FOUND
